## Integer Damath MCTS

In [1]:
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict, Any
import numpy as np
import copy
from collections import defaultdict, deque


In [2]:
Operator = Optional[str]  # '+', '-', 'x', '/' or None

@dataclass
class Piece:
    player: int  # 1 = Blue, -1 = Red
    value: int
    dama: bool = False

    def __repr__(self):
        # Convert B to blue emoji and R to red emoji for better visualization
        p = "🔵" if self.player == 1 else "🔴"
        val = f"{self.value:+d}" if self.value >= 0 else str(self.value)
        return f"{p}{'D' if self.dama else ''}{val}"
    
    
    def copy(self):
        return Piece(self.player, self.value, self.dama)

@dataclass
class Move:
    path: List[Tuple[int, int]]            # sequence of positions traversed
    captures: List[Tuple[int, int]]        # list of captured piece positions
    promotes: bool = False                 # whether the move results in promotion
    score_gain: int = 0                    # arithmetic reward from the move
    is_dama_capture: bool = False          # whether move made by dama
    is_multi_jump: bool = False            # whether multiple captures occurred

    def __repr__(self):
        cap_str = f" x{len(self.captures)}" if self.captures else ""
        promo = " (promo)" if self.promotes else ""
        return f"{self.path}{cap_str}{promo} +{self.score_gain}"



In [3]:
class DamathEnv:
    def __init__(self, rows=8, cols=8, operator_pattern=None):
        self.R = rows
        self.C = cols
        assert rows==8 and cols==8, "Currently implemented for 8x8 boards."
        # Operators on playable squares. Default pattern similar to provided image if None.
        if operator_pattern is None:
            operator_pattern = self.default_operator_board()
        self.op_board = operator_pattern
        # Piece board: dict (r,c)->Piece
        self.pieces: Dict[Tuple[int,int], Piece] = {}
        # Scores cumulative per player
        self.scores = {1: 0.0, -1: 0.0}
        self.to_move = 1  # 1 starts (blue on top)
        self.history_states = deque(maxlen=50)  # for repetition detection (store simple board hashes)
        
        # initialize sample starting board if user wants. We'll provide a helper to set initial config.
        self.init_default_integer_setup()

    def default_operator_board(self):
        # Create operator layout (8x8) using a repeating pattern similar to the uploaded assets.
        # Operators placed on playable squares (r+c)%2==1.
        ops = ['x','/','-','+']  # cycle
        board = [[None for _ in range(self.C)] for __ in range(self.R)]
        for r in range(self.R):
            for c in range(self.C):
                if (r + c) % 2 == 1:
                    # choose operator based on some pattern; rotate every cell
                    board[r][c] = ops[(r + 2*c) % len(ops)]
                else:
                    board[r][c] = None
        return board

    def init_default_integer_setup(self):
        # Initialize pieces according to the "integer damath" sample. We'll follow a symmetric-ish layout.
        # Blue (player=1) on top three rows playable squares, Red (player=-1) on bottom three rows.
        self.pieces = {}
        # sample integer values, you can customize to exact image mapping
        blue_values = [
            [-11, 8, -5, 2],
            [0, -3, 10, -7],
            [-9, 6, -1, 4],
        ]
        red_values = [
            [4, -1, 6, -9],
            [-7, 10, -3, 0],
            [2, -5, 8, -11]
        ]
        # place on playable squares; for top rows choose columns 0,2,4,6 for row 0,1.. pattern.
        # We'll place blues on rows 0..2 and reds on rows 5..7 so that they face each other.
        # map values left to right
        def playable_positions_on_row(r):
            # playable cols where (r+c)%2==1
            return [c for c in range(self.C) if (r+c)%2==1]
        # Blue top 3 rows
        for i, r in enumerate(range(0,3)):
            cols = playable_positions_on_row(r)
            vals = blue_values[i]
            for j, c in enumerate(cols[:len(vals)]):
                self.pieces[(r,c)] = Piece(player=1, value=vals[j], dama=False)
        # Red bottom 3 rows
        for i, r in enumerate(range(5,8)):
            cols = playable_positions_on_row(r)
            vals = red_values[i-0] if i < len(red_values) else []
            for j, c in enumerate(cols[:len(vals)]):
                self.pieces[(r,c)] = Piece(player=-1, value=vals[j], dama=False)
        # reset scores and to_move
        self.scores = {1:0.0, -1:0.0}
        self.to_move = 1
        self.history_states.clear()
        self.record_state()

    def copy(self):
        newenv = DamathEnv(self.R, self.C)
        newenv.op_board = copy.deepcopy(self.op_board)
        newenv.pieces = {k: v.copy() for k,v in self.pieces.items()}
        newenv.scores = dict(self.scores)
        newenv.to_move = self.to_move
        newenv.history_states = copy.deepcopy(self.history_states)
        return newenv

    def in_bounds(self, r,c):
        return 0 <= r < self.R and 0 <= c < self.C

    def is_playable(self, r,c):
        return self.in_bounds(r,c) and ((r+c)%2==1)

    def get_piece(self, r,c) -> Optional[Piece]:
        return self.pieces.get((r,c))

    def remove_piece(self, r,c):
        if (r,c) in self.pieces:
            del self.pieces[(r,c)]

    def move_piece(self, from_rc, to_rc):
        p = self.pieces.pop(from_rc)
        self.pieces[to_rc] = p
        return p

    def record_state(self):
        # Simple hash of pieces positions and values and to_move for repetition detection
        items = tuple(sorted([ (pos, piece.player, piece.value, piece.dama) for pos,piece in self.pieces.items() ]))
        key = (self.to_move, items)
        self.history_states.append(key)

    # ----------------------------- Operators & arithmetic -----------------------------
    def op_at(self, r,c):
        if not self.is_playable(r,c):
            return None
        return self.op_board[r][c]

    def apply_operator(self, op: str, a: int, b: int):
        if op == '+':
            return a + b
        if op == '-':
            return a - b
        if op == 'x' or op == 'X' or op == '*':
            return a * b
        if op == '/':
            # integer division semantics: handle division by zero and prefer integer division rounding toward zero
            if b == 0:
                # define a penalty or large negative? For now, return 0 to avoid crash.
                return 0
            return int(a / b)
        raise ValueError("Unknown op "+str(op))

    # ----------------------------- Movement and capture generation -----------------------------
    def generate_all_moves(self, player:int):
        """
        Returns a list of Move objects representing all legal moves for player.
        Enforces mandatory captures and priority rules described in prompt.
        """
        # 1) Find all capture sequences for every piece
        capture_moves = []
        for pos, piece in list(self.pieces.items()):
            if piece.player != player: continue
            caps = self._generate_captures_from(pos, piece)
            capture_moves.extend(caps)
        if len(capture_moves) > 0:
            # enforce capture priority: (1) max captures, (2) if tie, dama priority, (3) if regular has more captures than dama, regular wins
            max_cap = max(len(m.captures) for m in capture_moves)
            # filter moves with max captures
            maxcap_moves = [m for m in capture_moves if len(m.captures)==max_cap]
            # among tied moves, if any are from dama pieces and any from regular, prefer dama (unless regular has strictly more captures in other moves)
            # but since we already filtered to max_cap, only need to prefer dama among ties => prefer moves with piece.dama True if any
            if any(self.pieces[m.path[0]].dama for m in maxcap_moves): # note m.path[0] original square
                maxcap_moves = [m for m in maxcap_moves if self.pieces[m.path[0]].dama]
            # compute score_gain for each move using current op board and multipliers
            for m in maxcap_moves:
                m.score_gain = self._compute_move_score(m, player)
            return maxcap_moves
        # 2) If no captures, generate simple moves including dama moves
        simple_moves = []
        for pos, piece in list(self.pieces.items()):
            if piece.player != player: continue
            sms = self._generate_simple_from(pos, piece)
            simple_moves.extend(sms)
        # mark score_gain zero for these
        for m in simple_moves:
            m.score_gain = 0.0
        return simple_moves

    def _generate_simple_from(self, pos, piece:Piece):
        r,c = pos
        moves = []
        if piece.dama:
            # dama can move any distance along diagonals (like king in international draughts)
            for dr,dc in [(-1,-1),(-1,1),(1,-1),(1,1)]:
                step=1
                while True:
                    nr = r + dr*step; nc = c + dc*step
                    if not self.in_bounds(nr,nc) or not self.is_playable(nr,nc): break
                    if (nr,nc) in self.pieces: break
                    path = [(r,c),(nr,nc)]
                    promotes = self._check_promotion(nr, piece.player)
                    moves.append(Move(path=path, captures=[], promotes=promotes))
                    step += 1
        else:
            # regular piece: forward-only? In Damath, regular pieces can move diagonally forward one space.
            # We assume player=1 moves 'down' (increasing row), player=-1 moves 'up' (decreasing row).
            dr = 1 if piece.player==1 else -1
            for dc in (-1,1):
                nr = r + dr; nc = c + dc
                if not self.in_bounds(nr,nc) or not self.is_playable(nr,nc): continue
                if (nr,nc) in self.pieces: continue
                path=[(r,c),(nr,nc)]
                promotes = self._check_promotion(nr, piece.player)
                moves.append(Move(path=path, captures=[], promotes=promotes))
        return moves

    def _generate_captures_from(self, pos, piece:Piece):
        # returns all capture sequences starting from this piece (as Move objects)
        # For regular pieces: jump over adjacent opponent piece landing on square beyond if empty; can chain.
        # For dama pieces: long-range capture along diagonals: can jump over an opponent piece that has at least one empty landing square beyond it on the same diagonal. Dama can land on any empty square beyond the captured piece on that diagonal (but rules about priority when multiple captures available are handled globally).
        results = []
        r,c = pos

        if piece.dama:
            # long-range captures: for each diagonal, find opponent pieces and possible landing squares beyond.
            for dr,dc in [(-1,-1),(-1,1),(1,-1),(1,1)]:
                # step along diagonal to find first opponent piece(s)
                step=1
                while True:
                    mr = r + dr*step; mc = c + dc*step
                    if not self.in_bounds(mr,mc) or not self.is_playable(mr,mc): break
                    if (mr,mc) in self.pieces:
                        target = self.pieces[(mr,mc)]
                        if target.player == piece.player:
                            break  # blocked by own piece
                        # find landing squares beyond (must be empty)
                        land_step = 1
                        while True:
                            lr = mr + dr*land_step; lc = mc + dc*land_step
                            if not self.in_bounds(lr,lc) or not self.is_playable(lr,lc): break
                            if (lr,lc) in self.pieces: break
                            # found a possible landing square
                            # create a tentative move: capture that one piece and land on (lr,lc)
                            new_env = self.copy()
                            # perform capture on new_env to continue searching for multi-captures
                            captured_piece = new_env.pieces.pop((mr,mc))
                            moved_piece = new_env.pieces.pop((r,c))
                            new_env.pieces[(lr,lc)] = moved_piece
                            # Recurse to find further captures from (lr,lc)
                            # Note: we store the captured piece object snapshot as part of capture tuple
                            further = new_env._generate_captures_from((lr,lc), moved_piece)
                            if len(further)==0:
                                m = Move(path=[(r,c),(lr,lc)], captures=[(mr,mc,captured_piece)], promotes=False)
                                results.append(m)
                            else:
                                for fm in further:
                                    # prepend current capture to fm
                                    m = Move(path=[(r,c)] + fm.path, captures=[(mr,mc,captured_piece)] + fm.captures, promotes=False)
                                    results.append(m)
                            land_step += 1
                        break  # only consider the first opponent piece along diagonal for long-range capture
                    else:
                        step += 1

        else:
            # regular piece captures: check adjacent diagonals for opponent piece and landing square beyond
            for dr,dc in [(-1,-1),(-1,1),(1,-1),(1,1)]:
                ar = r + dr; ac = c + dc  # adjacent
                lr = r + 2*dr; lc = c + 2*dc  # landing beyond
                if not self.in_bounds(ar,ac) or not self.is_playable(ar,ac): continue
                if (ar,ac) not in self.pieces: continue
                if self.pieces[(ar,ac)].player == piece.player: continue
                if not self.in_bounds(lr,lc) or not self.is_playable(lr,lc): continue
                if (lr,lc) in self.pieces: continue
                # simulate capture and recurse
                new_env = self.copy()
                captured_piece = new_env.pieces.pop((ar,ac))
                moved_piece = new_env.pieces.pop((r,c))
                new_env.pieces[(lr,lc)] = moved_piece
                further = new_env._generate_captures_from((lr,lc), moved_piece)
                if len(further)==0:
                    m = Move(path=[(r,c),(lr,lc)], captures=[(ar,ac,captured_piece)], promotes=self._check_promotion(lr,piece.player))
                    results.append(m)
                else:
                    for fm in further:
                        m = Move(path=[(r,c)] + fm.path, captures=[(ar,ac,captured_piece)] + fm.captures, promotes=self._check_promotion(fm.path[-1][0], piece.player))
                        results.append(m)
        # remove duplicate sequences (same path) - dedupe by path and captured coordinates
        uniq = {}
        for m in results:
            key = (tuple(m.path), tuple((r,c,cp.value) for r,c,cp in m.captures))
            if key not in uniq or len(uniq[key].captures) < len(m.captures):
                uniq[key] = m
        return list(uniq.values())

    def _compute_move_score(self, move:Move, mover_player:int):
        # compute arithmetic score gain to mover for a capture move following rules:
        # - each captured piece adds op(own_value, captured_value) where op is operator on the landing square of that jump
        # - for dama captures, double score; if both dama, quadruple for that take.
        # - if a dama is taken by regular, the score is doubled as well (we handle multiplier on capture event)
        total = 0.0
        # need to reconstruct the piece value used in each jump: the moving piece's value at that time can be assumed unchanged
        # We'll use the mover's piece's original value
        # For multi-jumps, landing squares determine operators used
        mover_start = move.path[0]
        mover_piece = self.pieces.get(mover_start)
        mover_is_dama = mover_piece.dama if mover_piece else False
        # We must approximate the moving piece's value; in rules it's the mover's own piece value each time
        mover_value = mover_piece.value if mover_piece else 0
        # For each capture in sequence, determine operator at landing square (the square after the specific jump)
        # To find landing square for ith capture: it's path[1+i]
        for i, (cap_r,cap_c,cap_piece) in enumerate(move.captures):
            landing = move.path[1+i] if len(move.path) > 1+i else move.path[-1]
            op = self.op_at(*landing)
            # apply operator: op(self_value, captured_value)
            base = self.apply_operator(op, mover_value, cap_piece.value)
            # multipliers:
            mult = 1
            # If mover is dama, double for that take; if captured is dama and mover is dama -> quadruple (2*2)
            if mover_is_dama and cap_piece.dama:
                mult = 4
            elif mover_is_dama:
                mult = 2
            elif cap_piece.dama:
                # captured is dama and mover is regular: score doubled as well per rules
                mult = 2
            total += base * mult
        return total

    def _check_promotion(self, r, player):
        # If a piece reaches opposing end (row 7 for player=1, row 0 for player=-1), it is promoted
        if player==1 and r==self.R-1: return True
        if player==-1 and r==0: return True
        return False

    # ----------------------------- Apply Move -----------------------------
    def apply_move(self, move: "Move", verbose=False):
        """
        Apply a move and update board state + cumulative self.scores.
        Immediate reward is disabled — returns 0.0 every call.
        """
        player = self.to_move

        if len(move.captures) == 0:
            frm, to = move.path[0], move.path[-1]
            piece = self.pieces.pop(frm)
            self.pieces[to] = piece

            if self._check_promotion(to[0], piece.player) and not piece.dama:
                piece.dama = True

        else:
            frm = move.path[0]
            mover = self.pieces.pop(frm)
            current_pos = frm
            total_gain = 0.0

            for i, (cap_r, cap_c, cap_piece_snapshot) in enumerate(move.captures):
                landing = move.path[1 + i] if 1 + i < len(move.path) else move.path[-1]
                op = self.op_at(*landing)
                captured_piece = self.pieces.pop((cap_r, cap_c))
                base = self.apply_operator(op, mover.value, captured_piece.value)

                mult = 1
                if mover.dama and captured_piece.dama:
                    mult = 4
                elif mover.dama or captured_piece.dama:
                    mult = 2

                total_gain += base * mult
                current_pos = landing

            self.pieces[current_pos] = mover
            if self._check_promotion(current_pos[0], mover.player) and not mover.dama:
                mover.dama = True
            self.scores[mover.player] += total_gain

        self.to_move *= -1
        self.record_state()

        if verbose:
            print(f"[Player {player}] applied move, scores={self.scores}")
        return 0.0
        
            # ----------------------------- Terminal Reward -----------------------------
    def compute_final_reward_for(self, player: int = 1, alpha: float = 0.7, K: float = 100.0):
        """
        Compute final combined reward for `player`:
            final_reward = α * binary_outcome + (1 - α) * tanh((score_diff)/K)
        Returns (final_reward, winner, final_scores)
        """
        final_scores, winner = self.final_scores_and_winner()
        p_score = final_scores.get(player, 0.0)
        o_score = final_scores.get(-player, 0.0)

        # Binary component
        if winner == player:
            binary_reward = 1.0
        elif winner == -player:
            binary_reward = -1.0
        else:
            binary_reward = 0.0

        # Normalized score difference
        norm_diff = float(np.tanh((p_score - o_score) / K))
        final_reward = float(alpha * binary_reward + (1 - alpha) * norm_diff)
        final_reward = float(np.clip(final_reward, -1.0, 1.0))
        return final_reward, winner, final_scores


    # ----------------------------- Game end and scoring -----------------------------
    def legal_moves_exist(self, player:int):
        return len(self.generate_all_moves(player))>0

    def game_over(self):
        # game over if current player to move has no moves or only one player's chips remain or repetition or stalemate
        if not self.legal_moves_exist(self.to_move):
            return True
        # if only chips of one player remain
        players_present = set(p.player for p in self.pieces.values())
        if len(players_present) <= 1:
            return True
        # repetition detection (simple): if last 6 states repeated pattern
        # For now, consider repetition if history has same state repeated >=4 times overall
        hist = list(self.history_states)
        if len(hist) >= 8:
            counts = defaultdict(int)
            for h in hist:
                counts[h] += 1
                if counts[h] >= 4:
                    return True
        return False

    def get_state_hash(self):
        """Get hashable state representation."""
        items = tuple(sorted([(pos, p.player, p.value, p.dama) 
                        for pos, p in self.pieces.items()]))
        return (self.to_move, items)
    
    def final_scores_and_winner(self):
        # Add remaining pieces to their player's cumulative scores (dama doubled)
        final_scores = dict(self.scores)
        for (r,c), piece in self.pieces.items():
            val = piece.value * (2 if piece.dama else 1)
            final_scores[piece.player] += val
        # Determine winner
        if final_scores[1] > final_scores[-1]:
            winner = 1
        elif final_scores[1] < final_scores[-1]:
            winner = -1
        else:
            winner = 0
        return final_scores, winner

    # ----------------------------- Utilities -----------------------------
    def print_board(self):
        # Create board grid with operators and pieces
        grid = [[" ." for _ in range(self.C)] for __ in range(self.R)]
        for y in range(self.R):
            for x in range(self.C):
                if not self.is_playable(y, x):
                    grid[y][x] = "##"
                else:
                    op = self.op_at(y, x)
                    grid[y][x] = f" {op}"
        # Place pieces
        for (y, x), piece in self.pieces.items():
            sym = '🔵' if piece.player == 1 else '🔴'
            if piece.dama:
                sym += 'K'
            grid[y][x] = f"{sym}{piece.value:02d}" if piece.value >= 0 else f"{sym}{piece.value}"

        # Print column headers
        print("\n     " + " ".join([f"{x:>4}" for x in range(self.C)]))
        print("     " + "----" * self.C)

        # Print from top (highest y) to bottom (y=0)
        for y in reversed(range(self.R)):
            row_str = " ".join(f"{cell:>4}" for cell in grid[y])
            print(f"{y:>2} | {row_str} | {y:>2}")

        print("     " + "----" * self.C)
        print("     " + " ".join([f"{x:>4}" for x in range(self.C)]))


In [4]:
# Operator layout: (y, x) format
# y=0 is bottom row, y=7 is top row

operator_pattern_official = [
    ['x', '-', '/', 'x', '-', '+', '+', 'x'],
    ['-', '/', '-', 'x', '-', '+', 'x', '-'],
    ['-', '+', '+', '+', 'x', 'x', '/', '+'],
    ['x', '+', '+', '-', 'x', '/', '+', 'x'],
    ['x', '-', '/', 'x', '-', '-', '+', 'x'],
    ['-', '/', 'x', 'x', '-', '+', 'x', '-'],
    ['-', 'x', '+', '+', 'x', 'x', '/', '+'],
    ['+', '/', '-', '-', 'x', '/', '+', 'x']
]

env = DamathEnv(rows = 8, cols = 8, operator_pattern=operator_pattern_official)
env.print_board()
moves = env.generate_all_moves(env.to_move)
print("\nAvailable moves for player", env.to_move, "->", len(moves))
for move in moves:
    start = move.path[0]
    end = move.path[-1]
    piece = env.pieces.get(start, None)
    if piece:
        piece_type = "Dama" if piece.dama else "Regular"
        # Swap (y, x) to (x, y) for readability
        print(f"{piece_type} {piece.value:+d} at ({start[1]}, {start[0]}) -> ({end[1]}, {end[0]}) | ΔScore: {move.score_gain:+.1f}")
    else:
        print(f"Unknown piece at ({start[1]}, {start[0]}) -> ({end[1]}, {end[0]}) | ΔScore: {move.score_gain:+.1f}")




        0    1    2    3    4    5    6    7
     --------------------------------
 7 |  🔴02   ##  🔴-5   ##  🔴08   ## 🔴-11   ## |  7
 6 |   ##  🔴-7   ##  🔴10   ##  🔴-3   ##  🔴00 |  6
 5 |  🔴04   ##  🔴-1   ##  🔴06   ##  🔴-9   ## |  5
 4 |   ##    -   ##    x   ##    -   ##    x |  4
 3 |    x   ##    +   ##    x   ##    +   ## |  3
 2 |   ##  🔵-9   ##  🔵06   ##  🔵-1   ##  🔵04 |  2
 1 |  🔵00   ##  🔵-3   ##  🔵10   ##  🔵-7   ## |  1
 0 |   ## 🔵-11   ##  🔵08   ##  🔵-5   ##  🔵02 |  0
     --------------------------------
        0    1    2    3    4    5    6    7

Available moves for player 1 -> 7
Regular -9 at (1, 2) -> (0, 3) | ΔScore: +0.0
Regular -9 at (1, 2) -> (2, 3) | ΔScore: +0.0
Regular +6 at (3, 2) -> (2, 3) | ΔScore: +0.0
Regular +6 at (3, 2) -> (4, 3) | ΔScore: +0.0
Regular -1 at (5, 2) -> (4, 3) | ΔScore: +0.0
Regular -1 at (5, 2) -> (6, 3) | ΔScore: +0.0
Regular +4 at (7, 2) -> (6, 3) | ΔScore: +0.0


In [5]:
operator_pattern_official = [
    ['x', '+', '/', '-', '-', '/', '+', 'x'],
    ['-', '/', '+', 'x', 'x', '+', '/', '-'],
    ['-', '/', '+', 'x', 'x', '+', '/', '-'],
    ['x', '+', '/', '-', '-', '/', '+', 'x'],
    ['x', '+', '/', '-', '-', '/', '+', 'x'],
    ['-', '/', '+', 'x', 'x', '+', '/', '-'],
    ['-', '/', '+', 'x', 'x', '+', '/', '-'],
    ['x', '/', '/', '-', '-', '/', '+', 'x']
]

In [6]:
# Print empty board with no pieces, just operators
env_empty = DamathEnv(rows = 8, cols = 8, operator_pattern=operator_pattern_official)
env_empty.pieces = {}  # clear pieces
env_empty.print_board()


        0    1    2    3    4    5    6    7
     --------------------------------
 7 |    x   ##    /   ##    -   ##    +   ## |  7
 6 |   ##    /   ##    x   ##    +   ##    - |  6
 5 |    -   ##    +   ##    x   ##    /   ## |  5
 4 |   ##    +   ##    -   ##    /   ##    x |  4
 3 |    x   ##    /   ##    -   ##    +   ## |  3
 2 |   ##    /   ##    x   ##    +   ##    - |  2
 1 |    -   ##    +   ##    x   ##    /   ## |  1
 0 |   ##    +   ##    -   ##    /   ##    x |  0
     --------------------------------
        0    1    2    3    4    5    6    7


In [11]:
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

# create a unique run folder
run_name = f"runs/damath_selfplay_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
writer = SummaryWriter(run_name)

In [ ]:
def env_factory():
    e = DamathEnv(operator_pattern=operator_pattern_official)
    e.init_default_integer_setup()
    return e

In [7]:
from collections import deque
import random

class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)

    def add(self, examples):
        """Add list of examples [(state, pi, z, mover), ...]"""
        self.buffer.extend(examples)

    def sample(self, batch_size):
        """Randomly sample a batch of examples"""
        batch = random.sample(self.buffer, batch_size)
        return batch

    def __len__(self):
        return len(self.buffer)


In [8]:
# --- MCTS + Self-play + Training blueprint for Integer Damath (with TensorBoard metrics) ---

import math, random, time, copy
from collections import defaultdict, deque
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

# ----------------------- Playable squares mapping ------------------------
PLAYABLE_POS = [(r, c) for r in range(8) for c in range(8) if (r + c) % 2 == 1]
POS_TO_IDX = {pos: i for i, pos in enumerate(PLAYABLE_POS)}
IDX_TO_POS = {i: pos for pos, i in POS_TO_IDX.items()}
ACTION_SIZE = len(PLAYABLE_POS) * len(PLAYABLE_POS)  # 32*32 = 1024


def move_to_index(move):
    start = move.path[0]
    end = move.path[-1]
    s_idx = POS_TO_IDX[start]
    e_idx = POS_TO_IDX[end]
    return s_idx * len(PLAYABLE_POS) + e_idx


def index_to_move_index_pair(idx):
    n = len(PLAYABLE_POS)
    s = idx // n
    e = idx % n
    return s, e


# ----------------------- State encoder ------------------------
def encode_state(env):
    # Channels: 0 Blue regular, 1 Blue dama, 2 Red regular, 3 Red dama,
    # 4 Blue values, 5 Red values, 6 operator, 7 turn, 8 score diff
    C = 9
    H = 8
    W = 8
    state = np.zeros((C, H, W), dtype=np.float32)
    # pieces and values
    for (y, x), piece in env.pieces.items():
        if piece.player == 1:
            if piece.dama:
                state[1, y, x] = 1.0
            else:
                state[0, y, x] = 1.0
            state[4, y, x] = piece.value / 12.0
        else:
            if piece.dama:
                state[3, y, x] = 1.0
            else:
                state[2, y, x] = 1.0
            state[5, y, x] = piece.value / 12.0
    # operators
    op_map = {'+': 0.25, '-': 0.5, 'x': 0.75, 'X': 0.75, '*': 0.75, '/': 1.0, '÷': 1.0}
    for y in range(8):
        for x in range(8):
            if env.is_playable(y, x):
                state[6, y, x] = op_map.get(env.op_board[y][x], 0.0)
    # turn
    state[7, :, :] = 1.0 if env.to_move == 1 else 0.0
    # score diff
    blue = env.scores.get(1, 0.0)
    red = env.scores.get(-1, 0.0)
    state[8, :, :] = (blue - red) / 100.0
    return state


# ----------------------- Policy-Value Network ------------------------
class PVNet(nn.Module):
    def __init__(self, in_ch=9, board_h=8, board_w=8, action_size=ACTION_SIZE):
        super().__init__()
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.conv1 = nn.Conv2d(in_ch, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(64)
        # policy head
        self.policy_conv = nn.Conv2d(64, 32, kernel_size=1)
        self.policy_fc = nn.Linear(32 * board_h * board_w, action_size)
        # value head
        self.value_conv = nn.Conv2d(64, 16, kernel_size=1)
        self.value_fc1 = nn.Linear(16 * board_h * board_w, 64)
        self.value_fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        p = F.relu(self.policy_conv(x))
        p = p.view(p.size(0), -1)
        p = self.policy_fc(p)
        v = F.relu(self.value_conv(x))
        v = v.view(v.size(0), -1)
        v = F.relu(self.value_fc1(v))
        v = torch.tanh(self.value_fc2(v)).squeeze(-1)
        return p, v


# ----------------------- MCTS Engine ------------------------
class MCTSNode:
    def __init__(self, parent=None, prior=0.0):
        self.parent = parent
        self.children = {}
        self.N = 0
        self.W = 0.0
        self.P = prior
        self.Q = 0.0


class MCTS:
    def __init__(self, net, cpuct=1.0, num_sims=160, S=20):
        self.net = net
        self.cpuct = cpuct
        self.num_sims = num_sims
        self.S = S
        self.root = None
        self.nodes = {}

    def hash_state(self, env):
        items = tuple(sorted([(pos, p.player, p.value, p.dama) for pos, p in env.pieces.items()]))
        return (env.to_move, items)

    def run(self, root_env, add_noise=True):
        root_hash = self.hash_state(root_env)
        if self.root is None or getattr(self, "root_hash", None) != root_hash:
            self.root = MCTSNode(parent=None, prior=1.0)
            self.root_hash = root_hash

        for _ in range(self.num_sims):
            node = self.root
            env_sim = root_env.copy()
            path = []
            # Selection
            while node.children and not env_sim.game_over():
                best_score = -1e9
                best_act = None
                sqrtN = math.sqrt(node.N) if node.N > 0 else 1.0
                for a_idx, child in node.children.items():
                    U = self.cpuct * child.P * (sqrtN / (1 + child.N))
                    score = child.Q + U
                    if score > best_score:
                        best_score = score
                        best_act = a_idx
                        best_child = child
                if best_child is None:
                    break
                s_idx, e_idx = index_to_move_index_pair(best_act)
                start = IDX_TO_POS[s_idx]
                end = IDX_TO_POS[e_idx]
                legal = env_sim.generate_all_moves(env_sim.to_move)
                match = next((m for m in legal if m.path[0] == start and m.path[-1] == end), None)
                if match is None:
                    break
                env_sim.apply_move(match)
                path.append((node, best_act))
                node = best_child

            # Expansion and Evaluation
            if not env_sim.game_over():
                legal = env_sim.generate_all_moves(env_sim.to_move)
                if legal:
                    st = torch.tensor(encode_state(env_sim), dtype=torch.float32).unsqueeze(0).to(self.net.device)
                    with torch.no_grad():
                        logits, value = self.net(st)
                    logits = logits.squeeze(0).detach().cpu().numpy()
                    value = float(value.detach().cpu().item())
                    priors = {move_to_index(m): math.exp(logits[move_to_index(m)]) for m in legal}
                    ssum = sum(priors.values()) or 1.0
                    for k in priors:
                        priors[k] /= ssum
                    for idx, p in priors.items():
                        if idx not in node.children:
                            node.children[idx] = MCTSNode(parent=node, prior=p)
                else:
                    final_scores, _ = env_sim.final_scores_and_winner()
                    root_player = root_env.to_move
                    sd = final_scores[root_player] - final_scores[-root_player]
                    value = float(np.tanh(sd / self.S))
            else:
                final_scores, _ = env_sim.final_scores_and_winner()
                root_player = root_env.to_move
                sd = final_scores[root_player] - final_scores[-root_player]
                value = float(np.tanh(sd / self.S))
            self._backpropagate(node, value)
        visits = {a: c.N for a, c in self.root.children.items()}
        total = sum(visits.values()) or 1
        pi = {a: n / total for a, n in visits.items()}
        return pi

    def _backpropagate(self, node, value):
        v = value
        cur = node
        while cur is not None:
            cur.N += 1
            cur.W += v
            cur.Q = cur.W / cur.N
            v = -v
            cur = cur.parent


def self_play_game(env_factory, mcts, max_moves=300):
    """
    Runs one self-play game between two MCTS-controlled agents.
    Rewards are computed only at the end of the game using the environment's
    combined binary + normalized score difference reward system.
    """
    env = env_factory()
    states, pis, movers = [], [], []
    move_count = 0

    while not env.game_over() and move_count < max_moves:
        pi = mcts.run(env, add_noise=True)
        temp = 1.0 if move_count < 8 else 0.1
        act_idx = select_action_from_pi(pi, temp, EXPLORATION_RATE)

        # Encode current state
        state = encode_state(env)
        pi_full = np.zeros(ACTION_SIZE, dtype=np.float32)
        for a, p in pi.items():
            pi_full[a] = p

        states.append(state)
        pis.append(pi_full)
        movers.append(env.to_move)

        # Convert index to move
        s_idx, e_idx = index_to_move_index_pair(act_idx)
        start = IDX_TO_POS[s_idx]
        end = IDX_TO_POS[e_idx]
        legal = env.generate_all_moves(env.to_move)
        chosen = next((m for m in legal if m.path[0] == start and m.path[-1] == end),
                      random.choice(legal))

        env.apply_move(chosen)
        move_count += 1

    # === Terminal phase ===
    final_reward_p1, winner, final_scores = env.compute_final_reward_for(player=1)
    final_reward_p2, _, _ = env.compute_final_reward_for(player=-1)

    # Assign same reward to all states for each mover’s perspective
    zs = np.array([
        final_reward_p1 if m == 1 else final_reward_p2
        for m in movers
    ], dtype=np.float32)

    episode = {
        "states": np.array(states, dtype=np.float32),
        "policies": np.array(pis, dtype=np.float32),
        "values": zs,
        "movers": np.array(movers, dtype=np.int8),
        "final_scores": final_scores,
        "winner": winner,
        "move_count": move_count,
        "reward_p1": final_reward_p1,
        "reward_p2": final_reward_p2,
    }
    return episode

def select_action_from_pi(pi, temp=1.0, epsilon=0.05):
    acts = list(pi.keys())
    probs = np.array(list(pi.values()), dtype=np.float64)
    probs = np.nan_to_num(probs)
    probs = np.clip(probs, 0.0, None)

    # epsilon-greedy exploration
    if random.random() < epsilon:
        return random.choice(acts)

    if temp <= 1e-6:
        return acts[np.argmax(probs)]
    probs = probs ** (1.0 / (temp + 1e-12))
    probs /= probs.sum() or 1.0
    return np.random.choice(acts, p=probs)

# ----------------------- Network Training Function ------------------------
def train_network(net, optimizer, examples, batch_size=64, epochs=4):
    """
    Trains the policy-value network using self-play examples.
    Each example: (state, pi_target, z_target, mover)
    """
    net.train()
    device = net.device

    states = torch.tensor(np.array([ex[0] for ex in examples]), dtype=torch.float32).to(device)
    pis = torch.tensor(np.array([ex[1] for ex in examples]), dtype=torch.float32).to(device)
    zs = torch.tensor(np.array([ex[2] for ex in examples]), dtype=torch.float32).to(device)
    movers = torch.tensor(np.array([ex[3] for ex in examples]), dtype=torch.float32).to(device)

    dataset_size = len(examples)
    idxs = np.arange(dataset_size)

    for epoch in range(epochs):
        np.random.shuffle(idxs)
        total_policy_loss = 0.0
        total_value_loss = 0.0

        for start in range(0, dataset_size, batch_size):
            end = start + batch_size
            batch_idx = idxs[start:end]
            batch_states = states[batch_idx]
            batch_pis = pis[batch_idx]
            batch_zs = zs[batch_idx]
            batch_movers = movers[batch_idx]

            optimizer.zero_grad()
            p_logits, v_pred = net(batch_states)
            v_pred = v_pred.squeeze(-1)

            # Align target z with player's perspective
            # (if mover = -1, flip the sign of reward)
            aligned_z = batch_zs * batch_movers

            # Losses
            value_loss = F.mse_loss(v_pred, aligned_z)
            policy_loss = -(batch_pis * F.log_softmax(p_logits, dim=1)).sum(dim=1).mean()

            loss = value_loss + policy_loss
            loss.backward()
            optimizer.step()

            total_policy_loss += policy_loss.item()
            total_value_loss += value_loss.item()

        print(f"    Epoch {epoch+1}/{epochs} | Policy Loss: {total_policy_loss:.3f} | Value Loss: {total_value_loss:.3f}")



In [ ]:
# ----------------------- TensorBoard Initialization ------------------------
run_name = f"runs/damath_selfplay_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
writer = SummaryWriter(run_name)
print(f"📈 Logging to TensorBoard: {run_name}")

# ----------------------- Hyperparameters ------------------------
NUM_ITERATIONS = 20
GAMES_PER_ITER = 50
MAX_MOVES_PER_GAME = 200
LR = 0.01
GAMMA = 0.9 # Discount factor for rewards
EXPLORATION_RATE = 0.05
BATCH_SIZE = 256
EPOCH = 5

# ----------------------- Environment & Network Setup ------------------------
net = PVNet().to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
optimizer = optim.Adam(net.parameters(), lr=LR)
mcts = MCTS(net, cpuct=1.0, num_sims=150, S=10)

# ----------------------- Training Loop ------------------------
# ----------------------- Training Loop (updated: per-player rewards logging) -----------------------
global_step = 0
reward_window_p1 = deque(maxlen=20)
reward_window_p2 = deque(maxlen=20)
replay_buffer = deque(maxlen=12000)
cumulative_wins = {1: 0, -1: 0, 0: 0}  # player1, player2, draws

for iteration in range(1, NUM_ITERATIONS + 1):
    all_examples = []
    game_total_rewards = []
    game_p1_rewards = []
    game_p2_rewards = []
    game_lengths = []
    score_diffs = []
    win_counts = {1: 0, -1: 0, 0: 0}
    start_time = time.time()

    for g in range(GAMES_PER_ITER):
        episode = self_play_game(env_factory, mcts, MAX_MOVES_PER_GAME)
        examples = list(zip(episode["states"], episode["policies"], episode["values"], episode["movers"]))
        final_scores = episode["final_scores"]
        winner = episode["winner"]
        reward_p1 = episode["reward_p1"]
        reward_p2 = episode["reward_p2"]

        all_examples.extend(examples)
        replay_buffer.extend(examples)

        final_p1 = final_scores.get(1, 0.0)
        final_p2 = final_scores.get(-1, 0.0)
        score_diff = final_p1 - final_p2

        game_p1_rewards.append(reward_p1)
        game_p2_rewards.append(reward_p2)
        game_total_rewards.append((reward_p1 + reward_p2) / 2)
        game_lengths.append(len(examples))
        score_diffs.append(score_diff)
        win_counts[winner] += 1
        cumulative_wins[winner] += 1

        reward_window_p1.append(reward_p1)
        reward_window_p2.append(reward_p2)

        # --- TensorBoard logging per game ---
        writer.add_scalar("Game/P1_Reward_Final", reward_p1, global_step)
        writer.add_scalar("Game/P2_Reward_Final", reward_p2, global_step)
        writer.add_scalar("Game/Mean Absolute Reward", abs(reward_p1), global_step)
        writer.add_scalar("Game/P1_FinalScore", final_p1, global_step)
        writer.add_scalar("Game/P2_FinalScore", final_p2, global_step)
        writer.add_scalar("Game/ScoreDiff", abs(score_diff), global_step)
        writer.add_scalar("Game/Length", len(examples), global_step)
        writer.add_scalar("Game/Winner", winner, global_step)
        writer.add_scalar("Win/P1_Cumulative", cumulative_wins[1], global_step)
        writer.add_scalar("Win/P2_Cumulative", cumulative_wins[-1], global_step)
        writer.add_scalar("Win/Draws_Cumulative", cumulative_wins[0], global_step)
        global_step += 1
        current_wins_p1 = cumulative_wins[1]
        current_wins_p2 = cumulative_wins[-1]
        writer.add_scalars("Win/Comparison", {"Player1": current_wins_p1, "Player2": current_wins_p2}, global_step)
        writer.add_scalar("WinRate/P1_Winrate", cumulative_wins[1]/global_step, global_step)
        writer.add_scalar("WinRate/P2_Winrate", cumulative_wins[-1]/global_step, global_step)
        writer.add_scalar("WinRate/Draws_Cumulative", cumulative_wins[0]/global_step, global_step)
        

        print(f"Iter {iteration} | Game {g+1}/{GAMES_PER_ITER} | "
            f"FinalScore P1={final_p1:.1f} | P2={final_p2:.1f} | "
            f"ScoreDiff={score_diff:+.1f} | FinalReward P1={reward_p1:+.3f} | "
            f"P2={reward_p2:+.3f} | Winner={winner} | Moves={len(examples)}")

    print(f"🏆 Cumulative Wins after Iter {iteration}: P1={cumulative_wins[1]}, P2={cumulative_wins[-1]}, Draws={cumulative_wins[0]}")

    # === Training Phase ===
    buffer_size = len(replay_buffer)
    sample_size = min(buffer_size, 2500)

    if buffer_size < 2500:
        print(f"⚠️ Replay buffer small ({buffer_size}), training on available samples.")
        continue
    else:
        print(f"Replay buffer size: {buffer_size} | Sampling {sample_size} for training.")

    sampled = random.sample(replay_buffer, sample_size)
    combined_examples = all_examples + sampled

    print(f"Training on {len(all_examples)} new + {sample_size} replay samples.")
    train_network(net, optimizer, combined_examples, batch_size=BATCH_SIZE, epochs=EPOCH)

    # === Aggregate metrics for the iteration ===
    mean_total_reward = abs(float(np.mean(game_total_rewards))) if game_total_rewards else 0.0
    mean_p1_reward = float(np.mean(game_p1_rewards)) if game_p1_rewards else 0.0
    mean_p2_reward = float(np.mean(game_p2_rewards)) if game_p2_rewards else 0.0
    rolling_p1 = float(np.mean(reward_window_p1)) if len(reward_window_p1) > 0 else 0.0
    rolling_p2 = float(np.mean(reward_window_p2)) if len(reward_window_p2) > 0 else 0.0
    mean_game_length = float(np.mean(game_lengths)) if game_lengths else 0.0
    mean_score_diff = abs(float(np.mean(score_diffs)) if score_diffs else 0.0)
    win_rate_p1 = win_counts[1] / GAMES_PER_ITER
    win_rate_p2 = win_counts[-1] / GAMES_PER_ITER
    # Log Cumulative Win Rate as well
    cumulative_win_rate_p1 = cumulative_wins[1] / global_step
    cumulative_win_rate_p2 = cumulative_wins[-1] / global_step

    # === Log to TensorBoard ===
    writer.add_scalar("Reward/Mean_Total", mean_total_reward, iteration)
    writer.add_scalar("Reward/Player1_Mean", mean_p1_reward, iteration)
    writer.add_scalar("Reward/Player2_Mean", mean_p2_reward, iteration)
    writer.add_scalar("Reward/Player1_Rolling20", rolling_p1, iteration)
    writer.add_scalar("Reward/Player2_Rolling20", rolling_p2, iteration)
    writer.add_scalars("Score/Final", {"Player1": final_p1, "Player2": final_p2}, global_step)
    writer.add_scalar("Game/AvgLength", mean_game_length, iteration)
    writer.add_scalar("Score/Differential", mean_score_diff, iteration)
    writer.add_scalars("WinRate", {"Player1": win_rate_p1, "Player2": win_rate_p2}, iteration)
    writer.add_scalars("CumulativeWinRate", {"Player1": cumulative_win_rate_p1, "Player2": cumulative_win_rate_p2}, iteration)

    print(f"✅ Iter {iteration} | Mean total R: {mean_total_reward:.3f} | "
          f"P1 mean: {mean_p1_reward:.3f} (roll20 {rolling_p1:.3f}) | "
          f"P2 mean: {mean_p2_reward:.3f} (roll20 {rolling_p2:.3f}) | "
          f"Avg len: {mean_game_length:.1f} | WinRate P1: {win_rate_p1:.2f}")

    # === Checkpoint & timing ===
    ckpt_path = f"damath_pvnet_iter{iteration:03d}.pth"
    torch.save(net.state_dict(), ckpt_path)
    writer.add_scalar("Time/Iteration", time.time() - start_time, iteration)

writer.close()
print("🎯 Training complete! View metrics with:")
print("   tensorboard --logdir=runs")


📈 Logging to TensorBoard: runs/damath_selfplay_20251018_015722
Iter 1 | Game 1/50 | FinalScore P1=-94.0 | P2=-49.0 | ScoreDiff=-45.0 | FinalReward P1=-0.827 | P2=+0.827 | Winner=-1 | Moves=58
Iter 1 | Game 2/50 | FinalScore P1=97.0 | P2=93.0 | ScoreDiff=+4.0 | FinalReward P1=+0.712 | P2=-0.712 | Winner=1 | Moves=41
Iter 1 | Game 3/50 | FinalScore P1=-16.0 | P2=9.0 | ScoreDiff=-25.0 | FinalReward P1=-0.773 | P2=+0.773 | Winner=-1 | Moves=38
Iter 1 | Game 4/50 | FinalScore P1=-1.0 | P2=33.0 | ScoreDiff=-34.0 | FinalReward P1=-0.798 | P2=+0.798 | Winner=-1 | Moves=30
Iter 1 | Game 5/50 | FinalScore P1=67.0 | P2=3.0 | ScoreDiff=+64.0 | FinalReward P1=+0.869 | P2=-0.869 | Winner=1 | Moves=45
Iter 1 | Game 6/50 | FinalScore P1=14.0 | P2=43.0 | ScoreDiff=-29.0 | FinalReward P1=-0.785 | P2=+0.785 | Winner=-1 | Moves=40
Iter 1 | Game 7/50 | FinalScore P1=-20.0 | P2=147.0 | ScoreDiff=-167.0 | FinalReward P1=-0.979 | P2=+0.979 | Winner=-1 | Moves=30
Iter 1 | Game 8/50 | FinalScore P1=-120.0 | P2=

In [20]:
# Save the model
final_model_path = "damath_pvnet_final_bestP1.pth"
torch.save(net.state_dict(), final_model_path)
print(f"💾 Final model saved to {final_model_path}")

💾 Final model saved to damath_pvnet_final_bestP1.pth


In [ ]:
# ----------------------- TensorBoard Initialization ------------------------
run_name = f"runs/damath_selfplay_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
writer = SummaryWriter(run_name)
print(f"📈 Logging to TensorBoard: {run_name}")

# ----------------------- Hyperparameters ------------------------
NUM_ITERATIONS = 20
GAMES_PER_ITER = 50
MAX_MOVES_PER_GAME = 200
LR = 0.01
GAMMA = 0.99 # Discount factor for rewards
EXPLORATION_RATE = 0.2
BATCH_SIZE = 256
EPOCH = 5

# ----------------------- Environment & Network Setup ------------------------
net = PVNet().to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
optimizer = optim.Adam(net.parameters(), lr=LR)
mcts = MCTS(net, cpuct=1.0, num_sims=150, S=10)

# ----------------------- Training Loop ------------------------
# ----------------------- Training Loop (updated: per-player rewards logging) -----------------------
global_step = 0
reward_window_p1 = deque(maxlen=20)
reward_window_p2 = deque(maxlen=20)
replay_buffer = deque(maxlen=12000)
cumulative_wins = {1: 0, -1: 0, 0: 0}  # player1, player2, draws

for iteration in range(1, NUM_ITERATIONS + 1):
    all_examples = []
    game_total_rewards = []
    game_p1_rewards = []
    game_p2_rewards = []
    game_lengths = []
    score_diffs = []
    win_counts = {1: 0, -1: 0, 0: 0}
    start_time = time.time()

    for g in range(GAMES_PER_ITER):
        episode = self_play_game(env_factory, mcts, MAX_MOVES_PER_GAME)
        examples = list(zip(episode["states"], episode["policies"], episode["values"], episode["movers"]))
        final_scores = episode["final_scores"]
        winner = episode["winner"]
        reward_p1 = episode["reward_p1"]
        reward_p2 = episode["reward_p2"]

        all_examples.extend(examples)
        replay_buffer.extend(examples)

        final_p1 = final_scores.get(1, 0.0)
        final_p2 = final_scores.get(-1, 0.0)
        score_diff = final_p1 - final_p2

        game_p1_rewards.append(reward_p1)
        game_p2_rewards.append(reward_p2)
        game_total_rewards.append((reward_p1 + reward_p2) / 2)
        game_lengths.append(len(examples))
        score_diffs.append(score_diff)
        win_counts[winner] += 1
        cumulative_wins[winner] += 1

        reward_window_p1.append(reward_p1)
        reward_window_p2.append(reward_p2)

        # --- TensorBoard logging per game ---
        writer.add_scalar("Game/P1_Reward_Final", reward_p1, global_step)
        writer.add_scalar("Game/P2_Reward_Final", reward_p2, global_step)
        writer.add_scalar("Game/Mean Absolute Reward", abs(reward_p1), global_step)
        writer.add_scalar("Game/P1_FinalScore", final_p1, global_step)
        writer.add_scalar("Game/P2_FinalScore", final_p2, global_step)
        writer.add_scalar("Game/ScoreDiff", abs(score_diff), global_step)
        writer.add_scalar("Game/Length", len(examples), global_step)
        writer.add_scalar("Game/Winner", winner, global_step)
        writer.add_scalar("Win/P1_Cumulative", cumulative_wins[1], global_step)
        writer.add_scalar("Win/P2_Cumulative", cumulative_wins[-1], global_step)
        writer.add_scalar("Win/Draws_Cumulative", cumulative_wins[0], global_step)
        global_step += 1
        current_wins_p1 = cumulative_wins[1]
        current_wins_p2 = cumulative_wins[-1]
        writer.add_scalars("Win/Comparison", {"Player1": current_wins_p1, "Player2": current_wins_p2}, global_step)
        writer.add_scalar("WinRate/P1_Winrate", cumulative_wins[1]/global_step, global_step)
        writer.add_scalar("WinRate/P2_Winrate", cumulative_wins[-1]/global_step, global_step)
        writer.add_scalar("WinRate/Draws_Cumulative", cumulative_wins[0]/global_step, global_step)
        

        print(f"Iter {iteration} | Game {g+1}/{GAMES_PER_ITER} | "
            f"FinalScore P1={final_p1:.1f} | P2={final_p2:.1f} | "
            f"ScoreDiff={score_diff:+.1f} | FinalReward P1={reward_p1:+.3f} | "
            f"P2={reward_p2:+.3f} | Winner={winner} | Moves={len(examples)}")

    print(f"🏆 Cumulative Wins after Iter {iteration}: P1={cumulative_wins[1]}, P2={cumulative_wins[-1]}, Draws={cumulative_wins[0]}")

    # === Training Phase ===
    buffer_size = len(replay_buffer)
    sample_size = min(buffer_size, 2500)

    if buffer_size < 2500:
        print(f"⚠️ Replay buffer small ({buffer_size}), training on available samples.")
        continue
    else:
        print(f"Replay buffer size: {buffer_size} | Sampling {sample_size} for training.")

    sampled = random.sample(replay_buffer, sample_size)
    combined_examples = all_examples + sampled

    print(f"Training on {len(all_examples)} new + {sample_size} replay samples.")
    train_network(net, optimizer, combined_examples, batch_size=BATCH_SIZE, epochs=EPOCH)

    # === Aggregate metrics for the iteration ===
    mean_total_reward = abs(float(np.mean(game_total_rewards))) if game_total_rewards else 0.0
    mean_p1_reward = float(np.mean(game_p1_rewards)) if game_p1_rewards else 0.0
    mean_p2_reward = float(np.mean(game_p2_rewards)) if game_p2_rewards else 0.0
    rolling_p1 = float(np.mean(reward_window_p1)) if len(reward_window_p1) > 0 else 0.0
    rolling_p2 = float(np.mean(reward_window_p2)) if len(reward_window_p2) > 0 else 0.0
    mean_game_length = float(np.mean(game_lengths)) if game_lengths else 0.0
    mean_score_diff = abs(float(np.mean(score_diffs)) if score_diffs else 0.0)
    win_rate_p1 = win_counts[1] / GAMES_PER_ITER
    win_rate_p2 = win_counts[-1] / GAMES_PER_ITER
    # Log Cumulative Win Rate as well
    cumulative_win_rate_p1 = cumulative_wins[1] / global_step
    cumulative_win_rate_p2 = cumulative_wins[-1] / global_step

    # === Log to TensorBoard ===
    writer.add_scalar("Reward/Mean_Total", mean_total_reward, iteration)
    writer.add_scalar("Reward/Player1_Mean", mean_p1_reward, iteration)
    writer.add_scalar("Reward/Player2_Mean", mean_p2_reward, iteration)
    writer.add_scalar("Reward/Player1_Rolling20", rolling_p1, iteration)
    writer.add_scalar("Reward/Player2_Rolling20", rolling_p2, iteration)
    writer.add_scalars("Score/Final", {"Player1": final_p1, "Player2": final_p2}, global_step)
    writer.add_scalar("Game/AvgLength", mean_game_length, iteration)
    writer.add_scalar("Score/Differential", mean_score_diff, iteration)
    writer.add_scalars("WinRate", {"Player1": win_rate_p1, "Player2": win_rate_p2}, iteration)
    writer.add_scalars("CumulativeWinRate", {"Player1": cumulative_win_rate_p1, "Player2": cumulative_win_rate_p2}, iteration)

    print(f"✅ Iter {iteration} | Mean total R: {mean_total_reward:.3f} | "
          f"P1 mean: {mean_p1_reward:.3f} (roll20 {rolling_p1:.3f}) | "
          f"P2 mean: {mean_p2_reward:.3f} (roll20 {rolling_p2:.3f}) | "
          f"Avg len: {mean_game_length:.1f} | WinRate P1: {win_rate_p1:.2f}")

    # === Checkpoint & timing ===
    ckpt_path = f"damath_pvnet_iter{iteration:03d}.pth"
    torch.save(net.state_dict(), ckpt_path)
    writer.add_scalar("Time/Iteration", time.time() - start_time, iteration)

writer.close()
print("🎯 Training complete! View metrics with:")
print("   tensorboard --logdir=runs")


📈 Logging to TensorBoard: runs/damath_selfplay_20251018_182058
Iter 1 | Game 1/50 | FinalScore P1=152.0 | P2=192.0 | ScoreDiff=-40.0 | FinalReward P1=-0.814 | P2=+0.814 | Winner=-1 | Moves=68
Iter 1 | Game 2/50 | FinalScore P1=204.0 | P2=14.0 | ScoreDiff=+190.0 | FinalReward P1=+0.987 | P2=-0.987 | Winner=1 | Moves=75
Iter 1 | Game 3/50 | FinalScore P1=-48.0 | P2=-24.0 | ScoreDiff=-24.0 | FinalReward P1=-0.771 | P2=+0.771 | Winner=-1 | Moves=52
Iter 1 | Game 4/50 | FinalScore P1=50.0 | P2=-6.0 | ScoreDiff=+56.0 | FinalReward P1=+0.852 | P2=-0.852 | Winner=1 | Moves=43
Iter 1 | Game 5/50 | FinalScore P1=73.0 | P2=-34.0 | ScoreDiff=+107.0 | FinalReward P1=+0.937 | P2=-0.937 | Winner=1 | Moves=53
Iter 1 | Game 6/50 | FinalScore P1=-48.0 | P2=19.0 | ScoreDiff=-67.0 | FinalReward P1=-0.875 | P2=+0.875 | Winner=-1 | Moves=68
Iter 1 | Game 7/50 | FinalScore P1=84.0 | P2=-30.0 | ScoreDiff=+114.0 | FinalReward P1=+0.944 | P2=-0.944 | Winner=1 | Moves=57
Iter 1 | Game 8/50 | FinalScore P1=16.0 |

In [22]:
# Save the model
final_model_path = "damath_pvnet_final_bestP2.pth"
torch.save(net.state_dict(), final_model_path)
print(f"💾 Final model saved to {final_model_path}")

💾 Final model saved to damath_pvnet_final_bestP2.pth


In [ ]:
# ----------------------- TensorBoard Initialization ------------------------
run_name = f"runs/damath_selfplay_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
writer = SummaryWriter(run_name)
print(f"📈 Logging to TensorBoard: {run_name}")

# ----------------------- Hyperparameters ------------------------
NUM_ITERATIONS = 20
GAMES_PER_ITER = 50
MAX_MOVES_PER_GAME = 200
LR = 0.001
GAMMA = 0.9 # Discount factor for rewards
EXPLORATION_RATE = 0.2
BATCH_SIZE = 256
EPOCH = 5

# ----------------------- Environment & Network Setup ------------------------
net = PVNet().to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
optimizer = optim.Adam(net.parameters(), lr=LR)
mcts = MCTS(net, cpuct=1.0, num_sims=150, S=10)

# ----------------------- Training Loop ------------------------
# ----------------------- Training Loop (updated: per-player rewards logging) -----------------------
global_step = 0
reward_window_p1 = deque(maxlen=20)
reward_window_p2 = deque(maxlen=20)
replay_buffer = deque(maxlen=12000)
cumulative_wins = {1: 0, -1: 0, 0: 0}  # player1, player2, draws

for iteration in range(1, NUM_ITERATIONS + 1):
    all_examples = []
    game_total_rewards = []
    game_p1_rewards = []
    game_p2_rewards = []
    game_lengths = []
    score_diffs = []
    win_counts = {1: 0, -1: 0, 0: 0}
    start_time = time.time()

    for g in range(GAMES_PER_ITER):
        episode = self_play_game(env_factory, mcts, MAX_MOVES_PER_GAME)
        examples = list(zip(episode["states"], episode["policies"], episode["values"], episode["movers"]))
        final_scores = episode["final_scores"]
        winner = episode["winner"]
        reward_p1 = episode["reward_p1"]
        reward_p2 = episode["reward_p2"]

        all_examples.extend(examples)
        replay_buffer.extend(examples)

        final_p1 = final_scores.get(1, 0.0)
        final_p2 = final_scores.get(-1, 0.0)
        score_diff = final_p1 - final_p2

        game_p1_rewards.append(reward_p1)
        game_p2_rewards.append(reward_p2)
        game_total_rewards.append((reward_p1 + reward_p2) / 2)
        game_lengths.append(len(examples))
        score_diffs.append(score_diff)
        win_counts[winner] += 1
        cumulative_wins[winner] += 1

        reward_window_p1.append(reward_p1)
        reward_window_p2.append(reward_p2)

        # --- TensorBoard logging per game ---
        writer.add_scalar("Game/P1_Reward_Final", reward_p1, global_step)
        writer.add_scalar("Game/P2_Reward_Final", reward_p2, global_step)
        writer.add_scalar("Game/Mean Absolute Reward", abs(reward_p1), global_step)
        writer.add_scalar("Game/P1_FinalScore", final_p1, global_step)
        writer.add_scalar("Game/P2_FinalScore", final_p2, global_step)
        writer.add_scalar("Game/ScoreDiff", abs(score_diff), global_step)
        writer.add_scalar("Game/Length", len(examples), global_step)
        writer.add_scalar("Game/Winner", winner, global_step)
        writer.add_scalar("Win/P1_Cumulative", cumulative_wins[1], global_step)
        writer.add_scalar("Win/P2_Cumulative", cumulative_wins[-1], global_step)
        writer.add_scalar("Win/Draws_Cumulative", cumulative_wins[0], global_step)
        global_step += 1
        current_wins_p1 = cumulative_wins[1]
        current_wins_p2 = cumulative_wins[-1]
        writer.add_scalars("Win/Comparison", {"Player1": current_wins_p1, "Player2": current_wins_p2}, global_step)
        writer.add_scalar("WinRate/P1_Winrate", cumulative_wins[1]/global_step, global_step)
        writer.add_scalar("WinRate/P2_Winrate", cumulative_wins[-1]/global_step, global_step)
        writer.add_scalar("WinRate/Draws_Cumulative", cumulative_wins[0]/global_step, global_step)
        

        print(f"Iter {iteration} | Game {g+1}/{GAMES_PER_ITER} | "
            f"FinalScore P1={final_p1:.1f} | P2={final_p2:.1f} | "
            f"ScoreDiff={score_diff:+.1f} | FinalReward P1={reward_p1:+.3f} | "
            f"P2={reward_p2:+.3f} | Winner={winner} | Moves={len(examples)}")

    print(f"🏆 Cumulative Wins after Iter {iteration}: P1={cumulative_wins[1]}, P2={cumulative_wins[-1]}, Draws={cumulative_wins[0]}")

    # === Training Phase ===
    buffer_size = len(replay_buffer)
    sample_size = min(buffer_size, 2500)

    if buffer_size < 2500:
        print(f"⚠️ Replay buffer small ({buffer_size}), training on available samples.")
        continue
    else:
        print(f"Replay buffer size: {buffer_size} | Sampling {sample_size} for training.")

    sampled = random.sample(replay_buffer, sample_size)
    combined_examples = all_examples + sampled

    print(f"Training on {len(all_examples)} new + {sample_size} replay samples.")
    train_network(net, optimizer, combined_examples, batch_size=BATCH_SIZE, epochs=EPOCH)

    # === Aggregate metrics for the iteration ===
    mean_total_reward = abs(float(np.mean(game_total_rewards))) if game_total_rewards else 0.0
    mean_p1_reward = float(np.mean(game_p1_rewards)) if game_p1_rewards else 0.0
    mean_p2_reward = float(np.mean(game_p2_rewards)) if game_p2_rewards else 0.0
    rolling_p1 = float(np.mean(reward_window_p1)) if len(reward_window_p1) > 0 else 0.0
    rolling_p2 = float(np.mean(reward_window_p2)) if len(reward_window_p2) > 0 else 0.0
    mean_game_length = float(np.mean(game_lengths)) if game_lengths else 0.0
    mean_score_diff = abs(float(np.mean(score_diffs)) if score_diffs else 0.0)
    win_rate_p1 = win_counts[1] / GAMES_PER_ITER
    win_rate_p2 = win_counts[-1] / GAMES_PER_ITER
    # Log Cumulative Win Rate as well
    cumulative_win_rate_p1 = cumulative_wins[1] / global_step
    cumulative_win_rate_p2 = cumulative_wins[-1] / global_step

    # === Log to TensorBoard ===
    writer.add_scalar("Reward/Mean_Total", mean_total_reward, iteration)
    writer.add_scalar("Reward/Player1_Mean", mean_p1_reward, iteration)
    writer.add_scalar("Reward/Player2_Mean", mean_p2_reward, iteration)
    writer.add_scalar("Reward/Player1_Rolling20", rolling_p1, iteration)
    writer.add_scalar("Reward/Player2_Rolling20", rolling_p2, iteration)
    writer.add_scalars("Score/Final", {"Player1": final_p1, "Player2": final_p2}, global_step)
    writer.add_scalar("Game/AvgLength", mean_game_length, iteration)
    writer.add_scalar("Score/Differential", mean_score_diff, iteration)
    writer.add_scalars("WinRate", {"Player1": win_rate_p1, "Player2": win_rate_p2}, iteration)
    writer.add_scalars("CumulativeWinRate", {"Player1": cumulative_win_rate_p1, "Player2": cumulative_win_rate_p2}, iteration)

    print(f"✅ Iter {iteration} | Mean total R: {mean_total_reward:.3f} | "
          f"P1 mean: {mean_p1_reward:.3f} (roll20 {rolling_p1:.3f}) | "
          f"P2 mean: {mean_p2_reward:.3f} (roll20 {rolling_p2:.3f}) | "
          f"Avg len: {mean_game_length:.1f} | WinRate P1: {win_rate_p1:.2f}")

    # === Checkpoint & timing ===
    ckpt_path = f"damath_pvnet_iter{iteration:03d}.pth"
    torch.save(net.state_dict(), ckpt_path)
    writer.add_scalar("Time/Iteration", time.time() - start_time, iteration)

writer.close()
print("🎯 Training complete! View metrics with:")
print("   tensorboard --logdir=runs")


📈 Logging to TensorBoard: runs/damath_selfplay_20251019_121904
Iter 1 | Game 1/50 | FinalScore P1=-57.0 | P2=54.0 | ScoreDiff=-111.0 | FinalReward P1=-0.941 | P2=+0.941 | Winner=-1 | Moves=66
Iter 1 | Game 2/50 | FinalScore P1=31.0 | P2=-204.0 | ScoreDiff=+235.0 | FinalReward P1=+0.995 | P2=-0.995 | Winner=1 | Moves=59
Iter 1 | Game 3/50 | FinalScore P1=107.0 | P2=-56.0 | ScoreDiff=+163.0 | FinalReward P1=+0.978 | P2=-0.978 | Winner=1 | Moves=57
Iter 1 | Game 4/50 | FinalScore P1=-27.0 | P2=-43.0 | ScoreDiff=+16.0 | FinalReward P1=+0.748 | P2=-0.748 | Winner=1 | Moves=39
Iter 1 | Game 5/50 | FinalScore P1=13.0 | P2=9.0 | ScoreDiff=+4.0 | FinalReward P1=+0.712 | P2=-0.712 | Winner=1 | Moves=55
Iter 1 | Game 6/50 | FinalScore P1=14.0 | P2=88.0 | ScoreDiff=-74.0 | FinalReward P1=-0.889 | P2=+0.889 | Winner=-1 | Moves=38
Iter 1 | Game 7/50 | FinalScore P1=-3.0 | P2=11.0 | ScoreDiff=-14.0 | FinalReward P1=-0.742 | P2=+0.742 | Winner=-1 | Moves=46
Iter 1 | Game 8/50 | FinalScore P1=26.0 | P2

In [24]:
# Save the model
final_model_path = "damath_pvnet_final_bestBalanced.pth"
torch.save(net.state_dict(), final_model_path)
print(f"💾 Final model saved to {final_model_path}")

💾 Final model saved to damath_pvnet_final_bestBalanced.pth


In [ ]:
# ----------------------- TensorBoard Initialization ------------------------
run_name = f"runs/damath_selfplay_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
writer = SummaryWriter(run_name)
print(f"📈 Logging to TensorBoard: {run_name}")

# ----------------------- Hyperparameters ------------------------
NUM_ITERATIONS = 20
GAMES_PER_ITER = 50
MAX_MOVES_PER_GAME = 200
LR = 0.01
GAMMA = 0.9 # Discount factor for rewards
EXPLORATION_RATE = 0.9
BATCH_SIZE = 256
EPOCH = 5

# ----------------------- Environment & Network Setup ------------------------
net = PVNet().to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
optimizer = optim.Adam(net.parameters(), lr=LR)
mcts = MCTS(net, cpuct=1.0, num_sims=150, S=10)

# ----------------------- Training Loop ------------------------
# ----------------------- Training Loop (updated: per-player rewards logging) -----------------------
global_step = 0
reward_window_p1 = deque(maxlen=20)
reward_window_p2 = deque(maxlen=20)
replay_buffer = deque(maxlen=12000)
cumulative_wins = {1: 0, -1: 0, 0: 0}  # player1, player2, draws

for iteration in range(1, NUM_ITERATIONS + 1):
    all_examples = []
    game_total_rewards = []
    game_p1_rewards = []
    game_p2_rewards = []
    game_lengths = []
    score_diffs = []
    win_counts = {1: 0, -1: 0, 0: 0}
    start_time = time.time()

    for g in range(GAMES_PER_ITER):
        episode = self_play_game(env_factory, mcts, MAX_MOVES_PER_GAME)
        examples = list(zip(episode["states"], episode["policies"], episode["values"], episode["movers"]))
        final_scores = episode["final_scores"]
        winner = episode["winner"]
        reward_p1 = episode["reward_p1"]
        reward_p2 = episode["reward_p2"]

        all_examples.extend(examples)
        replay_buffer.extend(examples)

        final_p1 = final_scores.get(1, 0.0)
        final_p2 = final_scores.get(-1, 0.0)
        score_diff = final_p1 - final_p2

        game_p1_rewards.append(reward_p1)
        game_p2_rewards.append(reward_p2)
        game_total_rewards.append((reward_p1 + reward_p2) / 2)
        game_lengths.append(len(examples))
        score_diffs.append(score_diff)
        win_counts[winner] += 1
        cumulative_wins[winner] += 1

        reward_window_p1.append(reward_p1)
        reward_window_p2.append(reward_p2)

        # --- TensorBoard logging per game ---
        writer.add_scalar("Game/P1_Reward_Final", reward_p1, global_step)
        writer.add_scalar("Game/P2_Reward_Final", reward_p2, global_step)
        writer.add_scalar("Game/Mean Absolute Reward", abs(reward_p1), global_step)
        writer.add_scalar("Game/P1_FinalScore", final_p1, global_step)
        writer.add_scalar("Game/P2_FinalScore", final_p2, global_step)
        writer.add_scalar("Game/ScoreDiff", abs(score_diff), global_step)
        writer.add_scalar("Game/Length", len(examples), global_step)
        writer.add_scalar("Game/Winner", winner, global_step)
        writer.add_scalar("Win/P1_Cumulative", cumulative_wins[1], global_step)
        writer.add_scalar("Win/P2_Cumulative", cumulative_wins[-1], global_step)
        writer.add_scalar("Win/Draws_Cumulative", cumulative_wins[0], global_step)
        global_step += 1
        current_wins_p1 = cumulative_wins[1]
        current_wins_p2 = cumulative_wins[-1]
        writer.add_scalars("Win/Comparison", {"Player1": current_wins_p1, "Player2": current_wins_p2}, global_step)
        writer.add_scalar("WinRate/P1_Winrate", cumulative_wins[1]/global_step, global_step)
        writer.add_scalar("WinRate/P2_Winrate", cumulative_wins[-1]/global_step, global_step)
        writer.add_scalar("WinRate/Draws_Cumulative", cumulative_wins[0]/global_step, global_step)
        

        print(f"Iter {iteration} | Game {g+1}/{GAMES_PER_ITER} | "
            f"FinalScore P1={final_p1:.1f} | P2={final_p2:.1f} | "
            f"ScoreDiff={score_diff:+.1f} | FinalReward P1={reward_p1:+.3f} | "
            f"P2={reward_p2:+.3f} | Winner={winner} | Moves={len(examples)}")

    print(f"🏆 Cumulative Wins after Iter {iteration}: P1={cumulative_wins[1]}, P2={cumulative_wins[-1]}, Draws={cumulative_wins[0]}")

    # === Training Phase ===
    buffer_size = len(replay_buffer)
    sample_size = min(buffer_size, 2500)

    if buffer_size < 2500:
        print(f"⚠️ Replay buffer small ({buffer_size}), training on available samples.")
        continue
    else:
        print(f"Replay buffer size: {buffer_size} | Sampling {sample_size} for training.")

    sampled = random.sample(replay_buffer, sample_size)
    combined_examples = all_examples + sampled

    print(f"Training on {len(all_examples)} new + {sample_size} replay samples.")
    train_network(net, optimizer, combined_examples, batch_size=BATCH_SIZE, epochs=EPOCH)

    # === Aggregate metrics for the iteration ===
    mean_total_reward = abs(float(np.mean(game_total_rewards))) if game_total_rewards else 0.0
    mean_p1_reward = float(np.mean(game_p1_rewards)) if game_p1_rewards else 0.0
    mean_p2_reward = float(np.mean(game_p2_rewards)) if game_p2_rewards else 0.0
    rolling_p1 = float(np.mean(reward_window_p1)) if len(reward_window_p1) > 0 else 0.0
    rolling_p2 = float(np.mean(reward_window_p2)) if len(reward_window_p2) > 0 else 0.0
    mean_game_length = float(np.mean(game_lengths)) if game_lengths else 0.0
    mean_score_diff = abs(float(np.mean(score_diffs)) if score_diffs else 0.0)
    win_rate_p1 = win_counts[1] / GAMES_PER_ITER
    win_rate_p2 = win_counts[-1] / GAMES_PER_ITER
    # Log Cumulative Win Rate as well
    cumulative_win_rate_p1 = cumulative_wins[1] / global_step
    cumulative_win_rate_p2 = cumulative_wins[-1] / global_step

    # === Log to TensorBoard ===
    writer.add_scalar("Reward/Mean_Total", mean_total_reward, iteration)
    writer.add_scalar("Reward/Player1_Mean", mean_p1_reward, iteration)
    writer.add_scalar("Reward/Player2_Mean", mean_p2_reward, iteration)
    writer.add_scalar("Reward/Player1_Rolling20", rolling_p1, iteration)
    writer.add_scalar("Reward/Player2_Rolling20", rolling_p2, iteration)
    writer.add_scalars("Score/Final", {"Player1": final_p1, "Player2": final_p2}, global_step)
    writer.add_scalar("Game/AvgLength", mean_game_length, iteration)
    writer.add_scalar("Score/Differential", mean_score_diff, iteration)
    writer.add_scalars("WinRate", {"Player1": win_rate_p1, "Player2": win_rate_p2}, iteration)
    writer.add_scalars("CumulativeWinRate", {"Player1": cumulative_win_rate_p1, "Player2": cumulative_win_rate_p2}, iteration)

    print(f"✅ Iter {iteration} | Mean total R: {mean_total_reward:.3f} | "
          f"P1 mean: {mean_p1_reward:.3f} (roll20 {rolling_p1:.3f}) | "
          f"P2 mean: {mean_p2_reward:.3f} (roll20 {rolling_p2:.3f}) | "
          f"Avg len: {mean_game_length:.1f} | WinRate P1: {win_rate_p1:.2f}")

    # === Checkpoint & timing ===
    ckpt_path = f"damath_pvnet_iter{iteration:03d}.pth"
    torch.save(net.state_dict(), ckpt_path)
    writer.add_scalar("Time/Iteration", time.time() - start_time, iteration)

writer.close()
print("🎯 Training complete! View metrics with:")
print("   tensorboard --logdir=runs")


📈 Logging to TensorBoard: runs/damath_selfplay_20251020_090440
Iter 1 | Game 1/50 | FinalScore P1=14.0 | P2=66.0 | ScoreDiff=-52.0 | FinalReward P1=-0.843 | P2=+0.843 | Winner=-1 | Moves=38
Iter 1 | Game 2/50 | FinalScore P1=-5.0 | P2=4.0 | ScoreDiff=-9.0 | FinalReward P1=-0.727 | P2=+0.727 | Winner=-1 | Moves=48
Iter 1 | Game 3/50 | FinalScore P1=-94.0 | P2=-36.0 | ScoreDiff=-58.0 | FinalReward P1=-0.857 | P2=+0.857 | Winner=-1 | Moves=69
Iter 1 | Game 4/50 | FinalScore P1=108.0 | P2=4.0 | ScoreDiff=+104.0 | FinalReward P1=+0.933 | P2=-0.933 | Winner=1 | Moves=77
Iter 1 | Game 5/50 | FinalScore P1=-68.0 | P2=-66.0 | ScoreDiff=-2.0 | FinalReward P1=-0.706 | P2=+0.706 | Winner=-1 | Moves=46
Iter 1 | Game 6/50 | FinalScore P1=-107.0 | P2=113.0 | ScoreDiff=-220.0 | FinalReward P1=-0.993 | P2=+0.993 | Winner=-1 | Moves=52
Iter 1 | Game 7/50 | FinalScore P1=-1.0 | P2=65.0 | ScoreDiff=-66.0 | FinalReward P1=-0.874 | P2=+0.874 | Winner=-1 | Moves=42
Iter 1 | Game 8/50 | FinalScore P1=-3.0 | P

C:\Users\Blix\AppData\Local\Temp\ipykernel_9724\2522934183.py:321: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  value_loss = F.mse_loss(v_pred, aligned_z)


    Epoch 4/5 | Policy Loss: 20.235 | Value Loss: 9.076
    Epoch 5/5 | Policy Loss: 22.462 | Value Loss: 11.558
✅ Iter 10 | Mean total R: 0.000 | P1 mean: 0.067 (roll20 0.101) | P2 mean: -0.067 (roll20 -0.101) | Avg len: 47.3 | WinRate P1: 0.54
Iter 11 | Game 1/50 | FinalScore P1=-17.0 | P2=27.0 | ScoreDiff=-44.0 | FinalReward P1=-0.824 | P2=+0.824 | Winner=-1 | Moves=41
Iter 11 | Game 2/50 | FinalScore P1=150.0 | P2=-44.0 | ScoreDiff=+194.0 | FinalReward P1=+0.988 | P2=-0.988 | Winner=1 | Moves=41
Iter 11 | Game 3/50 | FinalScore P1=30.0 | P2=-12.0 | ScoreDiff=+42.0 | FinalReward P1=+0.819 | P2=-0.819 | Winner=1 | Moves=39
Iter 11 | Game 4/50 | FinalScore P1=-22.0 | P2=61.0 | ScoreDiff=-83.0 | FinalReward P1=-0.904 | P2=+0.904 | Winner=-1 | Moves=55
Iter 11 | Game 5/50 | FinalScore P1=-35.0 | P2=66.0 | ScoreDiff=-101.0 | FinalReward P1=-0.930 | P2=+0.930 | Winner=-1 | Moves=44
Iter 11 | Game 6/50 | FinalScore P1=63.0 | P2=0.0 | ScoreDiff=+63.0 | FinalReward P1=+0.867 | P2=-0.867 | Wi

In [27]:
# Save the model
final_model_path = "damath_pvnet_final_highExploration.pth"
torch.save(net.state_dict(), final_model_path)
print(f"💾 Final model saved to {final_model_path}")

💾 Final model saved to damath_pvnet_final_highExploration.pth


In [29]:
# Evaluate net performance after training
device = net.device
net.eval()
with torch.no_grad():
    test_env = env_factory()
    test_state = torch.tensor(encode_state(test_env), dtype=torch.float32).unsqueeze(0).to(device)
    logits, value = net(test_state)
    probs = F.softmax(logits, dim=1).squeeze(0).cpu().numpy()
    value = value.item()
    print(f"Test state policy probs (top 5): {np.sort(probs)[-5:][::-1]}")
    print(f"Test state value estimate: {value:.3f}")
    

Test state policy probs (top 5): [0.89285636 0.0581157  0.02855609 0.00895495 0.00584532]
Test state value estimate: -0.037


In [9]:
# Import pickle
import pickle

In [10]:
class QLearner:
    def __init__(self, learning_rate=0.01, gamma=0.9, epsilon=0.2):
        """
        Initialize Q-Learning agent.
        
        Args:
            learning_rate: Step size for Q-value updates (0.01-0.1 typical)
            gamma: Discount factor for future rewards (0.9-0.99 typical)
            epsilon: Exploration rate for epsilon-greedy policy (0.1-0.3 typical)
        """
        self.Q = defaultdict(lambda: defaultdict(float))
        self.learning_rate = learning_rate
        self.gamma = gamma
        self.epsilon = epsilon
        self.update_count = 0  # Track number of updates for debugging
 
    def get_q_value(self, state, action):
        """Get Q-value for a state-action pair."""
        state_key = state
        action_key = self._move_to_key(action)
        return self.Q[state_key][action_key]

    def choose_action(self, env, moves):
        """
        Choose action using epsilon-greedy policy.
        
        With probability epsilon: random exploration
        Otherwise: greedy (best Q-value)
        
        Args:
            env: Game environment
            moves: List of legal moves
            
        Returns:
            Selected move
        """
        if not moves:
            return None
        
        # Exploration: random action
        if random.random() < self.epsilon:
            return random.choice(moves)
        
        # Exploitation: greedy action
        state = env.get_state_hash()
        q_values = [(self.get_q_value(state, m), m) for m in moves]
        max_q = max(q_values, key=lambda x: x[0])[0]
        best_moves = [m for q, m in q_values if q == max_q]
        return random.choice(best_moves)

    def update_q_value(self, state, action, target):
        """
        Update Q-value using Q-learning update rule:
        Q(s,a) ← Q(s,a) + α(target - Q(s,a))
        
        Where target is pre-computed as:
        - Terminal: immediate_reward + final_reward_bonus
        - Non-terminal: immediate_reward + γ*max_a'Q(s',a')
        
        Args:
            state: Current state
            action: Action taken
            target: Target Q-value (should be pre-calculated in training loop)
        """
        state_key = state
        action_key = self._move_to_key(action)
        
        current_q = self.Q[state_key][action_key]
        td_error = target - current_q
        self.Q[state_key][action_key] = current_q + self.learning_rate * td_error
        self.update_count += 1

    def _move_to_key(self, move):
        """
        Convert Move object to hashable key for Q-table indexing.
        
        Key consists of:
        - path: tuple of positions the piece moves through
        - captures: tuple of captured positions (with values for uniqueness)
        """
        if move is None:
            return None
        return (tuple(move.path), tuple((r, c) for r, c, _ in move.captures))

    def get_max_q_value(self, state, moves):
        """
        Get maximum Q-value across all available actions.
        Useful for bootstrapping in training loop.
        
        Args:
            state: Current state
            moves: List of available moves
            
        Returns:
            Maximum Q-value, or 0.0 if no moves
        """
        if not moves:
            return 0.0
        q_values = [self.get_q_value(state, m) for m in moves]
        return max(q_values) if q_values else 0.0

    def set_epsilon(self, epsilon):
        """Adjust exploration rate (for epsilon decay during training)."""
        self.epsilon = epsilon

    def set_learning_rate(self, learning_rate):
        """Adjust learning rate (for learning rate decay during training)."""
        self.learning_rate = learning_rate

    def get_stats(self):
        """Return training statistics."""
        total_states = len(self.Q)
        total_entries = sum(len(actions) for actions in self.Q.values())
        return {
            'total_states': total_states,
            'total_q_entries': total_entries,
            'total_updates': self.update_count,
            'avg_q_per_state': total_entries / total_states if total_states > 0 else 0
        }

    def save(self, path):
        """Save Q-table to file."""
        with open(path, 'wb') as f:
            pickle.dump({
                'Q': dict(self.Q),
                'learning_rate': self.learning_rate,
                'gamma': self.gamma,
                'epsilon': self.epsilon,
                'update_count': self.update_count
            }, f)
        print(f"Agent saved to {path}")

    def load(self, path):
        """Load Q-table from file."""
        with open(path, 'rb') as f:
            data = pickle.load(f)
            self.Q = defaultdict(lambda: defaultdict(float), data['Q'])
            self.learning_rate = data.get('learning_rate', self.learning_rate)
            self.gamma = data.get('gamma', self.gamma)
            self.epsilon = data.get('epsilon', self.epsilon)
            self.update_count = data.get('update_count', 0)
        print(f"Agent loaded from {path}")

    def reset_q_table(self):
        """Reset Q-table to initial state (useful for retraining)."""
        self.Q = defaultdict(lambda: defaultdict(float))
        self.update_count = 0
        print("Q-table reset")

In [11]:
# ----------------------- Q-Network ------------------------
class QNetwork(nn.Module):
    def __init__(self, in_ch=9, board_h=8, board_w=8, action_size=ACTION_SIZE):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.bn2 = nn.BatchNorm2d(128)
        self.bn3 = nn.BatchNorm2d(128)
        
        # Q-value head
        self.q_conv = nn.Conv2d(128, 64, kernel_size=1)
        self.q_fc1 = nn.Linear(64 * board_h * board_w, 512)
        self.q_fc2 = nn.Linear(512, action_size)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        
        q = F.relu(self.q_conv(x))
        q = q.view(q.size(0), -1)
        q = F.relu(self.q_fc1(q))
        q = self.q_fc2(q)
        return q

# ----------------------- Model Network ------------------------
class ModelNetwork(nn.Module):
    def __init__(self, in_ch=9, board_h=8, board_w=8, action_size=ACTION_SIZE):
        super().__init__()
        
        # State encoder
        self.state_conv1 = nn.Conv2d(in_ch, 64, kernel_size=3, padding=1)
        self.state_conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        
        # Action embedding
        self.action_embed = nn.Embedding(action_size, 128)
        
        # Combined prediction
        self.pred_conv1 = nn.Conv2d(256, 128, kernel_size=3, padding=1)
        self.pred_conv2 = nn.Conv2d(128, 64, kernel_size=3, padding=1)
        self.pred_conv3 = nn.Conv2d(64, in_ch, kernel_size=3, padding=1)
        
        # Reward prediction
        self.reward_fc1 = nn.Linear(128 * board_h * board_w, 256)
        self.reward_fc2 = nn.Linear(256, 1)

    def forward(self, state, action_idx):
        B = state.size(0)
        
        # Encode state
        s = F.relu(self.state_conv1(state))
        s = F.relu(self.state_conv2(s))
        
        # Encode action
        a = self.action_embed(action_idx)
        a = a.view(B, 128, 1, 1).expand(-1, -1, 8, 8)
        
        # Concatenate
        combined = torch.cat([s, a], dim=1)
        
        # Predict next state
        next_state = F.relu(self.pred_conv1(combined))
        next_state = F.relu(self.pred_conv2(next_state))
        next_state = self.pred_conv3(next_state)
        
        # Predict reward
        flat = s.view(B, -1)
        reward = F.relu(self.reward_fc1(flat))
        reward = self.reward_fc2(reward).squeeze(-1)
        
        return next_state, reward

In [159]:
class DynaQAgent:
    def __init__(self, q_net, model_net, model_optimizer, epsilon=0.1, alpha=0.001, 
                 gamma=0.95, planning_steps=10):
        """
        Args:
            q_net: Q-Network for value estimation
            model_net: Model Network for environment dynamics
            model_optimizer: Optimizer for the model network (IMPORTANT!)
            epsilon: Exploration rate
            alpha: Q-network learning rate
            gamma: Discount factor
            planning_steps: Number of planning steps per real step
        """
        self.q_net = q_net
        self.model_net = model_net
        self.model_optimizer = model_optimizer  # ✅ ADD THIS
        self.epsilon = epsilon
        self.alpha = alpha
        self.gamma = gamma
        self.planning_steps = planning_steps
        
        self.q_optimizer = optim.Adam(q_net.parameters(), lr=alpha)
        
        # Experience buffer
        self.experience_buffer = deque(maxlen=10000)

    def get_legal_mask(self, env):
        """Create mask for legal actions"""
        legal_moves = env.generate_all_moves(env.to_move)
        legal_indices = [move_to_index(m) for m in legal_moves]
        mask = np.zeros(ACTION_SIZE, dtype=np.float32)
        mask[legal_indices] = 1.0
        return mask, legal_moves, legal_indices

    def select_action(self, env, training=True):
        """Epsilon-greedy action selection"""
        mask, legal_moves, legal_indices = self.get_legal_mask(env)
        
        if not legal_moves:
            return None, None
        
        # Epsilon-greedy
        if training and random.random() < self.epsilon:
            chosen_idx = random.choice(legal_indices)
        else:
            state = encode_state(env)
            state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(next(self.q_net.parameters()).device)
            
            with torch.no_grad():
                q_values = self.q_net(state_tensor).squeeze(0).cpu().numpy()
            
            q_values = q_values * mask - (1 - mask) * 1e9
            chosen_idx = np.argmax(q_values)
        
        # Convert to move
        s_idx, e_idx = index_to_move_index_pair(chosen_idx)
        start = IDX_TO_POS[s_idx]
        end = IDX_TO_POS[e_idx]
        
        chosen_move = next((m for m in legal_moves if m.path[0] == start and m.path[-1] == end), 
                          random.choice(legal_moves))
        
        return chosen_move, chosen_idx

    def train_model_step(self, state, action_idx, reward, next_state):
        """Train model on single transition - called immediately after each step"""
        device = next(self.model_net.parameters()).device
        
        state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
        action_tensor = torch.tensor([action_idx], dtype=torch.long).to(device)
        next_state_tensor = torch.tensor(next_state, dtype=torch.float32).unsqueeze(0).to(device)
        reward_tensor = torch.tensor([reward], dtype=torch.float32).to(device)
        
        # Forward pass
        pred_next_state, pred_reward = self.model_net(state_tensor, action_tensor)
        
        # Calculate losses
        state_loss = F.mse_loss(pred_next_state, next_state_tensor)
        reward_loss = F.mse_loss(pred_reward, reward_tensor)
        loss = state_loss + reward_loss
        
        # Backward pass
        self.model_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model_net.parameters(), 1.0)
        self.model_optimizer.step()
        
        return loss.item()

    def update_q(self, state, action_idx, reward, next_state, done, next_legal_mask):
        """Q-learning update"""
        device = next(self.q_net.parameters()).device
        
        state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
        next_state_tensor = torch.tensor(next_state, dtype=torch.float32).unsqueeze(0).to(device)
        action_tensor = torch.tensor([action_idx], dtype=torch.long).to(device)
        reward_tensor = torch.tensor([reward], dtype=torch.float32).to(device)
        
        # Current Q
        current_q = self.q_net(state_tensor).gather(1, action_tensor.unsqueeze(1)).squeeze()
        
        # Target Q
        with torch.no_grad():
            if done:
                target_q = reward_tensor
            else:
                next_q_values = self.q_net(next_state_tensor).squeeze(0).cpu().numpy()
                next_q_values = next_q_values * next_legal_mask - (1 - next_legal_mask) * 1e9
                max_next_q = torch.tensor([np.max(next_q_values)], dtype=torch.float32).to(device)
                target_q = reward_tensor + self.gamma * max_next_q
        
        # Update
        loss = F.mse_loss(current_q, target_q)
        self.q_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_net.parameters(), 1.0)
        self.q_optimizer.step()
        
        return loss.item()

    def planning(self):
        """Dyna-Q planning step - use learned model to generate synthetic experience"""
        if len(self.experience_buffer) < 32:
            return 0.0
        
        total_loss = 0.0
        device = next(self.model_net.parameters()).device
        
        for _ in range(self.planning_steps):
            # Sample random experience
            state, action_idx, _, _, _ = random.choice(self.experience_buffer)
            
            state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
            action_tensor = torch.tensor([action_idx], dtype=torch.long).to(device)
            
            # Use model to predict next state and reward
            with torch.no_grad():
                pred_next_state, pred_reward = self.model_net(state_tensor, action_tensor)
                pred_next_state = pred_next_state.squeeze(0).cpu().numpy()
                pred_reward = pred_reward.item()
            
            # Update Q-network with synthetic experience
            dummy_mask = np.ones(ACTION_SIZE, dtype=np.float32)
            loss = self.update_q(state, action_idx, pred_reward, pred_next_state, False, dummy_mask)
            total_loss += loss
        
        return total_loss / self.planning_steps

    def store_experience(self, state, action_idx, reward, next_state, done):
        """Store experience and train model immediately"""
        self.experience_buffer.append((state, action_idx, reward, next_state, done))
        # ✅ Train model on this transition
        self.train_model_step(state, action_idx, reward, next_state)

In [160]:
class DamathEnv_QLearner:
    def __init__(self, rows=8, cols=8, operator_pattern=None):
        self.R = rows
        self.C = cols
        assert rows==8 and cols==8, "Currently implemented for 8x8 boards."
        
        # Operators on playable squares. Default pattern similar to provided image if None.
        if operator_pattern is None:
            operator_pattern = self.default_operator_board()
        self.op_board = operator_pattern
        
        # Piece board: dict (r,c)->Piece
        self.pieces: Dict[Tuple[int,int], Piece] = {}
        
        # Scores cumulative per player
        self.scores = {1: 0.0, -1: 0.0}
        self.to_move = 1  # 1 starts (blue on top)
        self.history_states = deque(maxlen=50)  # for repetition detection
        
        # Reward configuration for immediate rewards during gameplay
        self.reward_config = {
            'capture_multiplier': 1.0,      # scale immediate capture score
            'promotion_bonus': 10.0,        # bonus for promoting to dama
            'dama_bonus': 5.0,              # bonus for capturing dama pieces
            'piece_loss_penalty': -5.0,     # penalty for losing a piece (not currently used)
            'dama_loss_penalty': -15.0,     # penalty for losing a dama (not currently used)
        }
        
        # initialize sample starting board
        self.init_default_integer_setup()

    def default_operator_board(self):
        """Create operator layout (8x8) using a repeating pattern."""
        ops = ['x','/','-','+']  # cycle
        board = [[None for _ in range(self.C)] for __ in range(self.R)]
        for r in range(self.R):
            for c in range(self.C):
                if (r + c) % 2 == 1:
                    board[r][c] = ops[(r + 2*c) % len(ops)]
                else:
                    board[r][c] = None
        return board

    def init_default_integer_setup(self):
        """Initialize pieces with default values."""
        self.pieces = {}
        blue_values = [
            [-11, 8, -5, 2],
            [0, -3, 10, -7],
            [-9, 6, -1, 4],
        ]
        red_values = [
            [4, -1, 6, -9],
            [-7, 10, -3, 0],
            [2, -5, 8, -11]
        ]
        
        def playable_positions_on_row(r):
            return [c for c in range(self.C) if (r+c)%2==1]
        
        # Blue top 3 rows
        for i, r in enumerate(range(0,3)):
            cols = playable_positions_on_row(r)
            vals = blue_values[i]
            for j, c in enumerate(cols[:len(vals)]):
                self.pieces[(r,c)] = Piece(player=1, value=vals[j], dama=False)
        
        # Red bottom 3 rows
        for i, r in enumerate(range(5,8)):
            cols = playable_positions_on_row(r)
            vals = red_values[i-0] if i < len(red_values) else []
            for j, c in enumerate(cols[:len(vals)]):
                self.pieces[(r,c)] = Piece(player=-1, value=vals[j], dama=False)
        
        self.scores = {1:0.0, -1:0.0}
        self.to_move = 1
        self.history_states.clear()
        self.record_state()

    def copy(self):
        newenv = DamathEnv(self.R, self.C)
        newenv.op_board = copy.deepcopy(self.op_board)
        newenv.pieces = {k: v.copy() for k,v in self.pieces.items()}
        newenv.scores = dict(self.scores)
        newenv.to_move = self.to_move
        newenv.history_states = copy.deepcopy(self.history_states)
        newenv.reward_config = dict(self.reward_config)
        return newenv

    def in_bounds(self, r,c):
        return 0 <= r < self.R and 0 <= c < self.C

    def is_playable(self, r,c):
        return self.in_bounds(r,c) and ((r+c)%2==1)

    def get_piece(self, r,c) -> Optional[Piece]:
        return self.pieces.get((r,c))

    def remove_piece(self, r,c):
        if (r,c) in self.pieces:
            del self.pieces[(r,c)]

    def move_piece(self, from_rc, to_rc):
        p = self.pieces.pop(from_rc)
        self.pieces[to_rc] = p
        return p

    def record_state(self):
        """Record state for repetition detection."""
        items = tuple(sorted([ (pos, piece.player, piece.value, piece.dama) 
                               for pos,piece in self.pieces.items() ]))
        key = (self.to_move, items)
        self.history_states.append(key)

    def op_at(self, r,c):
        """Get operator at position (r,c)."""
        if not self.is_playable(r,c):
            return None
        return self.op_board[r][c]

    def apply_operator(self, op: str, a: int, b: int):
        """Apply operator to two numbers."""
        if op == '+':
            return a + b
        if op == '-':
            return a - b
        if op == 'x' or op == 'X' or op == '*':
            return a * b
        if op == '/':
            if b == 0:
                return 0
            return int(a / b)
        raise ValueError("Unknown op "+str(op))

    def generate_all_moves(self, player:int):
        """Generate all legal moves for player."""
        capture_moves = []
        for pos, piece in list(self.pieces.items()):
            if piece.player != player: 
                continue
            caps = self._generate_captures_from(pos, piece)
            capture_moves.extend(caps)
        
        if len(capture_moves) > 0:
            max_cap = max(len(m.captures) for m in capture_moves)
            maxcap_moves = [m for m in capture_moves if len(m.captures)==max_cap]
            if any(self.pieces[m.path[0]].dama for m in maxcap_moves):
                maxcap_moves = [m for m in maxcap_moves if self.pieces[m.path[0]].dama]
            for m in maxcap_moves:
                m.score_gain = self._compute_move_score(m, player)
            return maxcap_moves
        
        simple_moves = []
        for pos, piece in list(self.pieces.items()):
            if piece.player != player: 
                continue
            sms = self._generate_simple_from(pos, piece)
            simple_moves.extend(sms)
        
        for m in simple_moves:
            m.score_gain = 0.0
        return simple_moves

    def _generate_simple_from(self, pos, piece:Piece):
        """Generate simple (non-capture) moves."""
        r,c = pos
        moves = []
        
        if piece.dama:
            for dr,dc in [(-1,-1),(-1,1),(1,-1),(1,1)]:
                step=1
                while True:
                    nr = r + dr*step
                    nc = c + dc*step
                    if not self.in_bounds(nr,nc) or not self.is_playable(nr,nc): 
                        break
                    if (nr,nc) in self.pieces: 
                        break
                    path = [(r,c),(nr,nc)]
                    promotes = self._check_promotion(nr, piece.player)
                    moves.append(Move(path=path, captures=[], promotes=promotes))
                    step += 1
        else:
            dr = 1 if piece.player==1 else -1
            for dc in (-1,1):
                nr = r + dr
                nc = c + dc
                if not self.in_bounds(nr,nc) or not self.is_playable(nr,nc): 
                    continue
                if (nr,nc) in self.pieces: 
                    continue
                path=[(r,c),(nr,nc)]
                promotes = self._check_promotion(nr, piece.player)
                moves.append(Move(path=path, captures=[], promotes=promotes))
        
        return moves

    def _generate_captures_from(self, pos, piece:Piece):
        """Generate all capture sequences from position."""
        results = []
        r,c = pos

        if piece.dama:
            for dr,dc in [(-1,-1),(-1,1),(1,-1),(1,1)]:
                step=1
                while True:
                    mr = r + dr*step
                    mc = c + dc*step
                    if not self.in_bounds(mr,mc) or not self.is_playable(mr,mc): 
                        break
                    if (mr,mc) in self.pieces:
                        target = self.pieces[(mr,mc)]
                        if target.player == piece.player:
                            break
                        land_step = 1
                        while True:
                            lr = mr + dr*land_step
                            lc = mc + dc*land_step
                            if not self.in_bounds(lr,lc) or not self.is_playable(lr,lc): 
                                break
                            if (lr,lc) in self.pieces: 
                                break
                            new_env = self.copy()
                            captured_piece = new_env.pieces.pop((mr,mc))
                            moved_piece = new_env.pieces.pop((r,c))
                            new_env.pieces[(lr,lc)] = moved_piece
                            further = new_env._generate_captures_from((lr,lc), moved_piece)
                            
                            if len(further)==0:
                                m = Move(path=[(r,c),(lr,lc)], captures=[(mr,mc,captured_piece)], promotes=False)
                                results.append(m)
                            else:
                                for fm in further:
                                    m = Move(path=[(r,c)] + fm.path, captures=[(mr,mc,captured_piece)] + fm.captures, promotes=False)
                                    results.append(m)
                            land_step += 1
                        break
                    else:
                        step += 1
        else:
            for dr,dc in [(-1,-1),(-1,1),(1,-1),(1,1)]:
                ar = r + dr
                ac = c + dc
                lr = r + 2*dr
                lc = c + 2*dc
                if not self.in_bounds(ar,ac) or not self.is_playable(ar,ac): 
                    continue
                if (ar,ac) not in self.pieces: 
                    continue
                if self.pieces[(ar,ac)].player == piece.player: 
                    continue
                if not self.in_bounds(lr,lc) or not self.is_playable(lr,lc): 
                    continue
                if (lr,lc) in self.pieces: 
                    continue
                
                new_env = self.copy()
                captured_piece = new_env.pieces.pop((ar,ac))
                moved_piece = new_env.pieces.pop((r,c))
                new_env.pieces[(lr,lc)] = moved_piece
                further = new_env._generate_captures_from((lr,lc), moved_piece)
                
                if len(further)==0:
                    m = Move(path=[(r,c),(lr,lc)], captures=[(ar,ac,captured_piece)], promotes=self._check_promotion(lr,piece.player))
                    results.append(m)
                else:
                    for fm in further:
                        m = Move(path=[(r,c)] + fm.path, captures=[(ar,ac,captured_piece)] + fm.captures, promotes=self._check_promotion(fm.path[-1][0], piece.player))
                        results.append(m)
        
        uniq = {}
        for m in results:
            key = (tuple(m.path), tuple((r,c,cp.value) for r,c,cp in m.captures))
            if key not in uniq or len(uniq[key].captures) < len(m.captures):
                uniq[key] = m
        return list(uniq.values())

    def _compute_move_score(self, move:Move, mover_player:int):
        """Compute arithmetic score gain for a capture move."""
        total = 0.0
        mover_start = move.path[0]
        mover_piece = self.pieces.get(mover_start)
        mover_is_dama = mover_piece.dama if mover_piece else False
        mover_value = mover_piece.value if mover_piece else 0
        
        for i, (cap_r,cap_c,cap_piece) in enumerate(move.captures):
            landing = move.path[1+i] if len(move.path) > 1+i else move.path[-1]
            op = self.op_at(*landing)
            base = self.apply_operator(op, mover_value, cap_piece.value)
            
            mult = 1
            if mover_is_dama and cap_piece.dama:
                mult = 4
            elif mover_is_dama:
                mult = 2
            elif cap_piece.dama:
                mult = 2
            total += base * mult
        return total

    def _check_promotion(self, r, player):
        """Check if piece should be promoted to dama."""
        if player==1 and r==self.R-1: 
            return True
        if player==-1 and r==0: 
            return True
        return False

    def apply_move_with_penalties(self, move: "Move", verbose=False):
        """
        Apply move and return immediate reward WITH penalties for opponent.
        
        This version returns a dict with rewards for both players:
        - Current player: gets reward for captures/promotions
        - Other player: gets penalty if their pieces were captured
        
        Use this if you want to apply penalties in the training loop.
        """
        player = self.to_move
        opponent = -player
        rewards = {player: 0.0, opponent: 0.0}

        if len(move.captures) == 0:
            # Simple move
            frm, to = move.path[0], move.path[-1]
            piece = self.pieces.pop(frm)
            self.pieces[to] = piece

            if self._check_promotion(to[0], piece.player) and not piece.dama:
                piece.dama = True
                rewards[player] += self.reward_config['promotion_bonus']

        else:
            # Capture move
            frm = move.path[0]
            mover = self.pieces.pop(frm)
            current_pos = frm
            total_gain = 0.0

            for i, (cap_r, cap_c, cap_piece_snapshot) in enumerate(move.captures):
                landing = move.path[1 + i] if 1 + i < len(move.path) else move.path[-1]
                op = self.op_at(*landing)
                captured_piece = self.pieces.pop((cap_r, cap_c))
                base = self.apply_operator(op, mover.value, captured_piece.value)

                mult = 1
                if mover.dama and captured_piece.dama:
                    mult = 4
                elif mover.dama or captured_piece.dama:
                    mult = 2

                total_gain += base * mult
                
                # Mover's reward for capture
                if captured_piece.dama:
                    rewards[player] += self.reward_config['dama_bonus']
                
                # Opponent's penalty for being captured
                if captured_piece.dama:
                    rewards[opponent] += self.reward_config['dama_loss_penalty']
                else:
                    rewards[opponent] += self.reward_config['piece_loss_penalty']
                
                current_pos = landing

            self.pieces[current_pos] = mover
            if self._check_promotion(current_pos[0], mover.player) and not mover.dama:
                mover.dama = True
                rewards[player] += self.reward_config['promotion_bonus']
            
            self.scores[mover.player] += total_gain
            rewards[player] += total_gain * self.reward_config['capture_multiplier']

        self.to_move *= -1
        self.record_state()

        if verbose:
            print(f"[Player {player}] captured, rewards: P1={rewards[1]:+.2f}, P2={rewards[-1]:+.2f}")
        
        return rewards  # Returns dict: {1: reward_p1, -1: reward_p2}

    def legal_moves_exist(self, player:int):
        return len(self.generate_all_moves(player))>0

    def game_over(self):
        """Check if game is over."""
        if not self.legal_moves_exist(self.to_move):
            return True
        players_present = set(p.player for p in self.pieces.values())
        if len(players_present) <= 1:
            return True
        hist = list(self.history_states)
        if len(hist) >= 8:
            counts = defaultdict(int)
            for h in hist:
                counts[h] += 1
                if counts[h] >= 4:
                    return True
        return False

    def final_scores_and_winner(self):
        """Compute final scores and determine winner."""
        final_scores = dict(self.scores)
        for (r,c), piece in self.pieces.items():
            val = piece.value * (2 if piece.dama else 1)
            final_scores[piece.player] += val
        
        if final_scores[1] > final_scores[-1]:
            winner = 1
        elif final_scores[1] < final_scores[-1]:
            winner = -1
        else:
            winner = 0
        return final_scores, winner

    def print_board(self):
        """Print current board state."""
        grid = [[" ." for _ in range(self.C)] for __ in range(self.R)]
        for y in range(self.R):
            for x in range(self.C):
                if not self.is_playable(y, x):
                    grid[y][x] = "##"
                else:
                    op = self.op_at(y, x)
                    grid[y][x] = f" {op}"
        
        for (y, x), piece in self.pieces.items():
            sym = '🔵' if piece.player == 1 else '🔴'
            if piece.dama:
                sym += 'K'
            grid[y][x] = f"{sym}{piece.value:02d}" if piece.value >= 0 else f"{sym}{piece.value}"

        print("\n     " + " ".join([f"{x:>4}" for x in range(self.C)]))
        print("     " + "----" * self.C)

        for y in reversed(range(self.R)):
            row_str = " ".join(f"{cell:>4}" for cell in grid[y])
            print(f"{y:>2} | {row_str} | {y:>2}")

        print("     " + "----" * self.C)
        print("     " + " ".join([f"{x:>4}" for x in range(self.C)]))
        
    def get_state_hash(self):
        """Get hashable state representation."""
        items = tuple(sorted([(pos, p.player, p.value, p.dama) 
                        for pos, p in self.pieces.items()]))
        return (self.to_move, items)
    
    def set_reward_config(self, config_dict: dict):
        """Update reward configuration."""
        self.reward_config.update(config_dict)

In [191]:
import numpy as np
import math
import random
# NOTE: Ensure 'encode_state', 'index_to_move_index_pair', and 'IDX_TO_POS' 
# are available in the scope where this function is run.

def mcts_exploit_move_selection(mcts_engine, env):
    """
    Runs MCTS for a single move, selects the best move based on visit count (argmax pi),
    and converts the action index into the required Move object.
    """
    # 1. Run MCTS without added noise (exploitation for evaluation)
    pi = mcts_engine.run(env, add_noise=False)
    
    if not pi:
        return None
    
    # 2. Select the move index with the highest visit count (max pi)
    # This corresponds to using a temperature T=0 in selection_action_from_pi.
    best_action_idx = max(pi, key=pi.get)
    
    # 3. Convert action index back to a Move object
    s_idx, e_idx = index_to_move_index_pair(best_action_idx)
    start = IDX_TO_POS[s_idx]
    end = IDX_TO_POS[e_idx]
    
    # We need the legal moves to match the chosen start and end positions to the specific Move object
    legal_moves = env.generate_all_moves(env.to_move)
    
    # Return the specific Move object that matches the start and end positions
    chosen_move = next((m for m in legal_moves if m.path[0] == start and m.path[-1] == end), None)
    
    return chosen_move


def evaluate_agents(env_class, q_agent, dyna_agent, mcts_agent, num_games=100, player1_agent='q', player2_agent='dyna'):
    """
    Evaluates two agents (Q, DynaQ, or MCTS) against each other over multiple games.
    
    Args:
        env_class: The canonical environment class (e.g., DamathEnv).
        q_agent: The Q-Learner agent instance.
        dyna_agent: The Dyna-Q agent instance.
        mcts_agent: The MCTS engine instance (assumed to be the MCTS class).
        num_games: Number of games to play.
        player1_agent: Agent type playing as Player 1 (value=1). 'q', 'dyna', or 'mcts'.
        player2_agent: Agent type playing as Player 2 (value=-1). 'q', 'dyna', or 'mcts'.
    """
    
    # --- Agent Mapping ---
    agent_instances = {
        'q': q_agent,
        'dyna': dyna_agent,
        'mcts': mcts_agent, 
    }

    agent_type_names = {
        'q': 'Q-Learner',
        'dyna': 'Dyna-Q Agent',
        'mcts': 'MCTS Agent',
    }
    
    agent_map = {
        1: agent_instances.get(player1_agent),
        -1: agent_instances.get(player2_agent),
    }
    agent_name_map = {
        1: agent_type_names.get(player1_agent),
        -1: agent_type_names.get(player2_agent),
    }
    
    agent_results = {
        'q_wins': 0, 
        'dyna_wins': 0,
        'mcts_wins': 0,
        'draws': 0
    }
    
    if None in agent_map.values():
        raise ValueError("Invalid player agent type specified. Must be 'q', 'dyna', or 'mcts' for both players.")

    score_diffs = []
    game_lengths = []

    print(f"--- Starting Evaluation: P1 ({agent_name_map[1]}) vs P2 ({agent_name_map[-1]}) ---")
    
    # --------------------------------------------------------------------------------------
    
    for game_idx in range(num_games):
        if agent_map[1] is mcts_agent or agent_map[-1] is mcts_agent:
            mcts_agent.root = None
            mcts_agent.root_hash = None
            mcts_agent.nodes.clear()
        
        env = env_class() 
        move_count = 0

        while not env.game_over() and move_count < 300:
            current_player = env.to_move
            move = None
            
            current_agent = agent_map[current_player]

            # 1. Action Selection based on Agent Instance/Type
            if current_agent is q_agent:
                # Q-Agent: uses choose_action, returns single move
                legal_moves = env.generate_all_moves(current_player)
                if legal_moves:
                    move = q_agent.choose_action(env, legal_moves) 
            
            elif current_agent is dyna_agent:
                # Dyna-Q Agent: uses select_action, returns move and index (discard index)
                move, chosen_idx = dyna_agent.select_action(env, training=False)

            elif current_agent is mcts_agent:
                # MCTS Agent: Use the helper function for exploitation
                move = mcts_exploit_move_selection(mcts_agent, env)

            # 2. Game flow control
            if move is None:
                # Break if the agent couldn't find a move (or there are no legal moves)
                break 

            # 3. Apply the move
            env.apply_move(move) 
            move_count += 1

        # --- Game End & Logging ---
        final_scores, winner = env.final_scores_and_winner()
        
        # Log wins based on the agent type that won
        if winner != 0:
            winning_agent_instance = agent_map[winner]
            
            if winning_agent_instance is q_agent:
                winning_agent_type = 'q'
            elif winning_agent_instance is dyna_agent:
                winning_agent_type = 'dyna'
            elif winning_agent_instance is mcts_agent:
                winning_agent_type = 'mcts'
            else:
                winning_agent_type = 'draw' # Should not happen with validation
            
            agent_results[f'{winning_agent_type}_wins'] += 1
        else:
            agent_results['draws'] += 1
        
        # Log stats
        score_diff = abs(final_scores[1] - final_scores[-1])
        score_diffs.append(score_diff)
        game_lengths.append(move_count)

        print(f"Game {game_idx+1}/{num_games} | Winner: {agent_name_map.get(winner, 'Draw')} | Final Scores: {final_scores} | Score Diff: {score_diff:.2f} | Moves: {move_count}")

    # --- Final Results Summary ---
    total_games = num_games
    mean_score_diff = sum(score_diffs) / len(score_diffs) if score_diffs else 0
    mean_game_length = sum(game_lengths) / len(game_lengths) if game_lengths else 0
    
    print(f"\n--- Evaluation Results over {total_games} games: ---")
    print(f"P1 ({agent_name_map[1]}) vs P2 ({agent_name_map[-1]})")
    print("-" * 40)
    print(f"Q-Learner Wins: {agent_results['q_wins']} ({agent_results['q_wins']/total_games*100:.1f}%)")
    print(f"Dyna-Q Agent Wins: {agent_results['dyna_wins']} ({agent_results['dyna_wins']/total_games*100:.1f}%)")
    print(f"MCTS Agent Wins: {agent_results['mcts_wins']} ({agent_results['mcts_wins']/total_games*100:.1f}%)")
    print(f"Draws: {agent_results['draws']} ({agent_results['draws']/total_games*100:.1f}%)")
    print(f"Mean Absolute Score Difference: {mean_score_diff:.2f}")
    print(f"Mean Game Length (moves): {mean_game_length:.2f}")

    return agent_results, mean_score_diff, mean_game_length

In [219]:
# Load pretrained Q-Learner Agent
q_agent = QLearner(learning_rate=0.01, gamma=0.99, epsilon=0.2)
q_agent.load("C:\\Users\\Blix\\Documents\\GitHub\\CCRNFLRL_Integer_Damath\\Q-Learning_Models\\damath_agent_high-gamma.pkl")
# Print stats
stats = q_agent.get_stats()
print(f"Loaded Q-Learner Agent Stats: {stats}")

Agent loaded from C:\Users\Blix\Documents\GitHub\CCRNFLRL_Integer_Damath\Q-Learning_Models\damath_agent_high-gamma.pkl
Loaded Q-Learner Agent Stats: {'total_states': 38797, 'total_q_entries': 204523, 'total_updates': 48189, 'avg_q_per_state': 5.271618939608732}


In [220]:
def load_trained_agent(checkpoint_path, device=None):
     """Load a trained DynaQ agent from checkpoint"""
     if device is None:
         device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
     # Load checkpoint
     checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
     # Create networks
     q_net = QNetwork().to(device)
     model_net = ModelNetwork().to(device)
     # Load trained weights
     q_net.load_state_dict(checkpoint['q_net'])
     model_net.load_state_dict(checkpoint['model_net'])
     # Set to evaluation mode
     q_net.eval()
     model_net.eval()

     # Create optimizer (needed for agent initialization)
     model_optimizer = optim.Adam(model_net.parameters(), lr=0.01)

     # Create agent
     agent = DynaQAgent(
         q_net=q_net,
         model_net=model_net,
         model_optimizer=model_optimizer,
         epsilon=0.2,  # No exploration for evaluation
         alpha=0.01,
         gamma=0.5,
         planning_steps=10
     )

     print(f"✅ Loaded agent from {checkpoint_path}")
     if 'episodes_completed' in checkpoint:
         print(f"   Trained on {checkpoint['episodes_completed']} episodes")

     return agent, checkpoint

In [221]:
# Load pretrained Dyna-Q Network using load_trained_agent function
dyna_agent, dyna_checkpoint = load_trained_agent("C:\\Users\\Blix\\Documents\\GitHub\\CCRNFLRL_Integer_Damath\\DynaQ_Models\\DynaQ_LowDiscountFactor.pth")
# Set to eval mode
dyna_agent.q_net.eval()
dyna_agent.model_net.eval()

✅ Loaded agent from C:\Users\Blix\Documents\GitHub\CCRNFLRL_Integer_Damath\DynaQ_Models\DynaQ_LowDiscountFactor.pth


ModelNetwork(
  (state_conv1): Conv2d(9, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (state_conv2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (action_embed): Embedding(1024, 128)
  (pred_conv1): Conv2d(256, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pred_conv2): Conv2d(128, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pred_conv3): Conv2d(64, 9, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (reward_fc1): Linear(in_features=8192, out_features=256, bias=True)
  (reward_fc2): Linear(in_features=256, out_features=1, bias=True)
)

In [222]:
# Load PVNet from file path
q_net_path="C:\\Users\\Blix\\Documents\\GitHub\\CCRNFLRL_Integer_Damath\\damath_pvnet_final_LowLearningRate.pth"
q_net = PVNet()

# Load weights
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
q_net.load_state_dict(torch.load(q_net_path, map_location=device))
q_net.to(device)
# Create MCTS agent
mcts_agent = MCTS(
    q_net,
    cpuct=1.0,
    num_sims=150,
    S=10,
)
# Set to eval mode
mcts_agent.net.eval()


PVNet(
  (conv1): Conv2d(9, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn3): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (policy_conv): Conv2d(64, 32, kernel_size=(1, 1), stride=(1, 1))
  (policy_fc): Linear(in_features=2048, out_features=1024, bias=True)
  (value_conv): Conv2d(64, 16, kernel_size=(1, 1), stride=(1, 1))
  (value_fc1): Linear(in_features=1024, out_features=64, bias=True)
  (value_fc2): Linear(in_features=64, out_features=1, bias=True)
)

In [223]:
# Run function to evaluate both agents
evaluate_agents(env_factory, q_agent, dyna_agent, mcts_agent=mcts_agent, num_games=50, player1_agent='q', player2_agent='dyna')
evaluate_agents(env_factory, q_agent, dyna_agent, mcts_agent=mcts_agent, num_games=50, player1_agent='dyna', player2_agent='q')

--- Starting Evaluation: P1 (Q-Learner) vs P2 (Dyna-Q Agent) ---
Game 1/50 | Winner: Q-Learner | Final Scores: {1: 48.0, -1: -44.0} | Score Diff: 92.00 | Moves: 44
Game 2/50 | Winner: Dyna-Q Agent | Final Scores: {1: -35.0, -1: 67.0} | Score Diff: 102.00 | Moves: 54
Game 3/50 | Winner: Dyna-Q Agent | Final Scores: {1: -132.0, -1: 66.0} | Score Diff: 198.00 | Moves: 79
Game 4/50 | Winner: Dyna-Q Agent | Final Scores: {1: -124.0, -1: 44.0} | Score Diff: 168.00 | Moves: 43
Game 5/50 | Winner: Q-Learner | Final Scores: {1: 212.0, -1: 25.0} | Score Diff: 187.00 | Moves: 54
Game 6/50 | Winner: Dyna-Q Agent | Final Scores: {1: -175.0, -1: 19.0} | Score Diff: 194.00 | Moves: 40
Game 7/50 | Winner: Q-Learner | Final Scores: {1: 14.0, -1: -153.0} | Score Diff: 167.00 | Moves: 40
Game 8/50 | Winner: Dyna-Q Agent | Final Scores: {1: 2.0, -1: 60.0} | Score Diff: 58.00 | Moves: 46
Game 9/50 | Winner: Dyna-Q Agent | Final Scores: {1: 86.0, -1: 381.0} | Score Diff: 295.00 | Moves: 68
Game 10/50 | Winn

({'q_wins': 20, 'dyna_wins': 30, 'mcts_wins': 0, 'draws': 0}, 93.12, 48.36)

In [45]:
# Simulate a game between Dyna-Q and MCTS agents. Print board at every move done by the agent. Print the step done by each agent and the final summary of the game. Check score of each agent at the end of the game.
env = env_factory()
current_agent_map = {
    1: dyna_agent,   # Player 1: Dyna-Q Agent
    -1: mcts_agent  # Player 2: MCTS Agent
}
move_count = 0
while not env.game_over() and move_count < 300:
    current_player = env.to_move
    move = None
    
    current_agent = current_agent_map[current_player]

    # 1. Action Selection based on Agent Instance/Type
    if current_agent is dyna_agent:
        # Dyna-Q Agent: uses select_action, returns move and index (discard index)
        move, _ = dyna_agent.select_action(env, training=False)

    elif current_agent is mcts_agent:
        # MCTS Agent: Use the helper function for exploitation
        move = mcts_exploit_move_selection(mcts_agent, env)

    # 2. Game flow control
    if move is None:
        # Break if the agent couldn't find a move (or there are no legal moves)
        break 

    # 3. Apply the move
    print(f"\n--- Move {move_count+1} by Player {current_player} ({'Dyna-Q' if current_agent is dyna_agent else 'MCTS'}) ---")
    print(f"Move Path: {move.path}, Captures: {[ (r,c,p.value) for r,c,p in move.captures ]}, Promotes: {move.promotes}")
    env.apply_move(move) 
    env.print_board()
    move_count += 1

# --- Game End & Logging ---
final_scores, winner = env.final_scores_and_winner()
print(f"\n--- Game Over after {move_count} moves ---")
print(f"Final Scores: {final_scores}")



--- Move 1 by Player 1 (Dyna-Q) ---
Move Path: [(2, 7), (3, 6)], Captures: [], Promotes: False

        0    1    2    3    4    5    6    7
     --------------------------------
 7 |  🔴02   ##  🔴-5   ##  🔴08   ## 🔴-11   ## |  7
 6 |   ##  🔴-7   ##  🔴10   ##  🔴-3   ##  🔴00 |  6
 5 |  🔴04   ##  🔴-1   ##  🔴06   ##  🔴-9   ## |  5
 4 |   ##    +   ##    -   ##    /   ##    x |  4
 3 |    x   ##    /   ##    -   ##  🔵04   ## |  3
 2 |   ##  🔵-9   ##  🔵06   ##  🔵-1   ##    - |  2
 1 |  🔵00   ##  🔵-3   ##  🔵10   ##  🔵-7   ## |  1
 0 |   ## 🔵-11   ##  🔵08   ##  🔵-5   ##  🔵02 |  0
     --------------------------------
        0    1    2    3    4    5    6    7

--- Move 2 by Player -1 (MCTS) ---
Move Path: [(5, 2), (4, 3)], Captures: [], Promotes: False

        0    1    2    3    4    5    6    7
     --------------------------------
 7 |  🔴02   ##  🔴-5   ##  🔴08   ## 🔴-11   ## |  7
 6 |   ##  🔴-7   ##  🔴10   ##  🔴-3   ##  🔴00 |  6
 5 |  🔴04   ##    +   ##  🔴06   ##  🔴-9   ## |  5
 4 |   #

In [33]:
# Simulate a game from the trained network. Log what piece moved where and the score changes. Inverse the printing y-axis to match board representation.
net.eval()
mcts = MCTS(net, cpuct=1.0, num_sims=150, S=10)
env = env_factory()
move_count = 0

while not env.game_over() and move_count < 300:
    env.print_board()
    print(f"\nPlayer {env.to_move}'s turn. Current scores: {env.scores}")
    pi = mcts.run(env, add_noise=False)
    act_idx = select_action_from_pi(pi, temp=0.1)
    s_idx, e_idx = index_to_move_index_pair(act_idx)
    start = IDX_TO_POS[s_idx]; end = IDX_TO_POS[e_idx]
    legal = env.generate_all_moves(env.to_move)
    chosen = None
    for m in legal:
        if m.path[0]==start and m.path[-1]==end:
            chosen = m; break
    if chosen is None:
        chosen = random.choice(legal)
    # Print chosen move details and the piece that moved. Inverse (y,x) to (x,y) for readability
    piece = env.pieces.get(chosen.path[0], None)
    if piece:
        piece_type = "Dama" if piece.dama else "Regular"
        print(f"Chosen move: {piece_type} {piece.value:+d} at ({chosen.path[0][1]}, {chosen.path[0][0]}) -> ({chosen.path[-1][1]}, {chosen.path[-1][0]}) | ΔScore: {chosen.score_gain:+.1f}")
    env.apply_move(chosen, verbose=True)
    move_count += 1

# Print final board and result, rewards
env.print_board()
final_scores, winner = env.final_scores_and_winner()
print(f"\nGame Over! Final scores: {final_scores}, Winner: {winner}")
print(f"Total moves: {move_count}")
print(f"final reward for Player 1 (Blue): {final_scores.get(1,0) - final_scores.get(-1,0):.1f}")


        0    1    2    3    4    5    6    7
     --------------------------------
 7 |  🔴02   ##  🔴-5   ##  🔴08   ## 🔴-11   ## |  7
 6 |   ##  🔴-7   ##  🔴10   ##  🔴-3   ##  🔴00 |  6
 5 |  🔴04   ##  🔴-1   ##  🔴06   ##  🔴-9   ## |  5
 4 |   ##    +   ##    -   ##    /   ##    x |  4
 3 |    x   ##    /   ##    -   ##    +   ## |  3
 2 |   ##  🔵-9   ##  🔵06   ##  🔵-1   ##  🔵04 |  2
 1 |  🔵00   ##  🔵-3   ##  🔵10   ##  🔵-7   ## |  1
 0 |   ## 🔵-11   ##  🔵08   ##  🔵-5   ##  🔵02 |  0
     --------------------------------
        0    1    2    3    4    5    6    7

Player 1's turn. Current scores: {1: 0.0, -1: 0.0}
Chosen move: Regular +6 at (3, 2) -> (2, 3) | ΔScore: +0.0
[Player 1] applied move, scores={1: 0.0, -1: 0.0}

        0    1    2    3    4    5    6    7
     --------------------------------
 7 |  🔴02   ##  🔴-5   ##  🔴08   ## 🔴-11   ## |  7
 6 |   ##  🔴-7   ##  🔴10   ##  🔴-3   ##  🔴00 |  6
 5 |  🔴04   ##  🔴-1   ##  🔴06   ##  🔴-9   ## |  5
 4 |   ##    +   ##    -   ##    /   #